In [ ]:
import os
print(os.listdir("/kaggle/input/datasets/anushakirand/"))

In [ ]:
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from sklearn.metrics import (
    roc_auc_score, average_precision_score, precision_recall_curve,
    confusion_matrix, f1_score,
)

CHECKPOINT_PATH = f"/kaggle/working/checkpoints/{MODEL_NAME}_best.pt"

# Reuse from Cell 1 if present, else rebuild
if "val_df_g" in globals() and "test_df_g" in globals() and "id_to_path_g" in globals():
    print("Reusing splits from training cell.")
    val_df, test_df, id_to_path = val_df_g, test_df_g, id_to_path_g
    eval_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                              std=[0.229, 0.224, 0.225]),
    ])
    val_ds = ICHDataset(val_df, id_to_path, transform=eval_transform)
    test_ds = ICHDataset(test_df, id_to_path, transform=eval_transform)
else:
    print("Splits not found in memory — rebuilding (same random_state=42).")
    id_to_path = scan_input_dirs(INPUT_DIRS, MIN_FILE_SIZE_BYTES)
    df = load_labels(TRAIN_CSV)
    df = df[df.index.isin(id_to_path.keys())]
    _, val_df, test_df = make_splits(df)
    eval_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                              std=[0.229, 0.224, 0.225]),
    ])
    val_ds = ICHDataset(val_df, id_to_path, transform=eval_transform)
    test_ds = ICHDataset(test_df, id_to_path, transform=eval_transform)

val_loader = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

model = MODEL_BUILDERS[MODEL_NAME]()
checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()
print(f"Loaded checkpoint: epoch {checkpoint['epoch']}, val_loss={checkpoint['val_loss']:.4f}")


@torch.no_grad()
def run_inference(loader):
    all_probs, all_labels = [], []
    for imgs, labels in loader:
        imgs = imgs.to(DEVICE)
        with torch.autocast(device_type=DEVICE.type, enabled=USE_AMP):
            outputs = model(imgs)
        probs = torch.sigmoid(outputs.float()).cpu().numpy()
        all_probs.append(probs)
        all_labels.append(labels.numpy())
    return np.concatenate(all_probs), np.concatenate(all_labels)


print("Running inference on val split (threshold selection only)...")
val_probs, val_labels = run_inference(val_loader)
print("Running inference on test split (reported numbers)...")
test_probs, test_labels = run_inference(test_loader)
print(f"Done: {len(val_probs)} val images, {len(test_probs)} test images.")


def best_f1_threshold(y_true, y_prob):
    precisions, recalls, thresholds = precision_recall_curve(y_true, y_prob)
    f1s = 2 * precisions * recalls / (precisions + recalls + 1e-12)
    best_idx = np.nanargmax(f1s[:-1])
    return thresholds[best_idx] if len(thresholds) > 0 else 0.5


rows = []
confusion_matrices = {}

for i, name in enumerate(LABEL_COLS):
    y_true_val = val_labels[:, i]
    y_true_test = test_labels[:, i]
    y_prob_test = test_probs[:, i]

    if len(np.unique(y_true_val)) < 2 or len(np.unique(y_true_test)) < 2:
        continue

    thresh = best_f1_threshold(y_true_val, val_probs[:, i])
    y_pred_test = (y_prob_test >= thresh).astype(int)

    roc_auc = roc_auc_score(y_true_test, y_prob_test)
    pr_auc = average_precision_score(y_true_test, y_prob_test)

    tn, fp, fn, tp = confusion_matrix(y_true_test, y_pred_test, labels=[0, 1]).ravel()
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    f1 = f1_score(y_true_test, y_pred_test, zero_division=0)

    confusion_matrices[name] = np.array([[tn, fp], [fn, tp]])

    rows.append({
        "class": name, "roc_auc": roc_auc, "pr_auc": pr_auc,
        "threshold_from_val": thresh, "precision": precision,
        "recall_sensitivity": recall, "specificity": specificity, "f1": f1,
        "n_positive": int(y_true_test.sum()), "n_total": len(y_true_test),
    })

results_df = pd.DataFrame(rows).set_index("class")
macro_auc = roc_auc_score(test_labels, test_probs, average="macro")
micro_auc = roc_auc_score(test_labels, test_probs, average="micro")

print("\n" + "=" * 90)
print(f"PER-CLASS METRICS (test split) — {MODEL_NAME}, checkpoint epoch {checkpoint['epoch']}")
print("=" * 90)
print(results_df.round(4).to_string())
print(f"\nMacro-average ROC-AUC: {macro_auc:.4f}")
print(f"Micro-average ROC-AUC: {micro_auc:.4f}")

print("\n" + "=" * 90)
print("CONFUSION MATRICES (test split; rows=true, cols=predicted, order=[neg, pos])")
print("=" * 90)
for name, cm in confusion_matrices.items():
    print(f"\n{name}:")
    print(f"           pred_neg  pred_pos")
    print(f"true_neg   {cm[0,0]:>8}  {cm[0,1]:>8}")
    print(f"true_pos   {cm[1,0]:>8}  {cm[1,1]:>8}")

out_path = f"/kaggle/working/{MODEL_NAME}_eval_metrics.csv"
results_df.to_csv(out_path)
print(f"\nFull metrics table saved to {out_path}")

global val_probs_g, val_labels_g, test_probs_g, test_labels_g
val_probs_g, val_labels_g, test_probs_g, test_labels_g = val_probs, val_labels, test_probs, test_labels

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, precision_recall_curve, auc

# ------------------------------------------------------------------
# ROC + PR CURVES — per class, test split
# ------------------------------------------------------------------
# Note: these are computed once, not before/after temperature scaling.
# Temp scaling (logits / T) is a monotonic transform — it doesn't
# change the ranking of predictions, only their confidence, so ROC/PR
# curves (which are rank-based) are identical before and after. That's
# also why ROC-AUC/PR-AUC don't move in the calibration table.

n_classes = len(LABEL_COLS)
fig, axes = plt.subplots(2, n_classes, figsize=(4 * n_classes, 8))

for i, name in enumerate(LABEL_COLS):
    y_true = test_labels[:, i]
    y_prob = test_probs[:, i]

    if len(np.unique(y_true)) < 2:
        axes[0, i].set_visible(False)
        axes[1, i].set_visible(False)
        continue

    # ROC
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    roc_auc_val = auc(fpr, tpr)
    ax_roc = axes[0, i]
    ax_roc.plot(fpr, tpr, label=f"AUC = {roc_auc_val:.3f}")
    ax_roc.plot([0, 1], [0, 1], "k--", alpha=0.4)
    ax_roc.set_title(f"{name}\nROC curve")
    ax_roc.set_xlabel("False Positive Rate")
    ax_roc.set_ylabel("True Positive Rate")
    ax_roc.legend(fontsize=8, loc="lower right")

    # PR
    precision, recall, _ = precision_recall_curve(y_true, y_prob)
    pr_auc_val = average_precision_score(y_true, y_prob)
    base_rate = y_true.mean()
    ax_pr = axes[1, i]
    ax_pr.plot(recall, precision, label=f"AP = {pr_auc_val:.3f}")
    ax_pr.axhline(base_rate, color="k", linestyle="--", alpha=0.4,
                  label=f"baseline = {base_rate:.3f}")
    ax_pr.set_title(f"{name}\nPR curve")
    ax_pr.set_xlabel("Recall")
    ax_pr.set_ylabel("Precision")
    ax_pr.legend(fontsize=8, loc="upper right")

plt.suptitle(f"ROC & PR curves — {MODEL_NAME}, test split (n={len(test_labels)})", y=1.02)
plt.tight_layout()
plt.savefig(f"/kaggle/working/{MODEL_NAME}_roc_pr_curves.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Saved ROC/PR curve grid to /kaggle/working/{MODEL_NAME}_roc_pr_curves.png")

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

N_BINS = 15


class TemperatureScaler(nn.Module):
    def __init__(self):
        super().__init__()
        self.log_T = nn.Parameter(torch.zeros(1))

    def forward(self, logits):
        return logits / torch.exp(self.log_T)


@torch.no_grad()
def get_logits(loader):
    all_logits, all_labels = [], []
    for imgs, labels in loader:
        imgs = imgs.to(DEVICE)
        with torch.autocast(device_type=DEVICE.type, enabled=USE_AMP):
            outputs = model(imgs)
        all_logits.append(outputs.float().cpu())
        all_labels.append(labels)
    return torch.cat(all_logits), torch.cat(all_labels)


print("Running inference (logits) on val split (for fitting T)...")
val_logits, val_labels_t = get_logits(val_loader)
print("Running inference (logits) on test split (for reporting)...")
test_logits, test_labels_t = get_logits(test_loader)

scaler = TemperatureScaler()
optimizer = torch.optim.LBFGS([scaler.log_T], lr=0.05, max_iter=100)
bce = nn.BCEWithLogitsLoss()

def closure():
    optimizer.zero_grad()
    loss = bce(scaler(val_logits), val_labels_t)
    loss.backward()
    return loss

optimizer.step(closure)
T = torch.exp(scaler.log_T).item()
print(f"\nFitted temperature: T = {T:.4f}  "
      f"({'softens' if T > 1 else 'sharpens'} the model's confidence)")


def expected_and_max_calibration_error(y_true, y_prob, n_bins=15):
    bin_edges = np.linspace(0, 1, n_bins + 1)
    ece, mce = 0.0, 0.0
    bin_accs, bin_confs, bin_counts = [], [], []
    for lo, hi in zip(bin_edges[:-1], bin_edges[1:]):
        mask = (y_prob > lo) & (y_prob <= hi)
        count = mask.sum()
        if count == 0:
            bin_accs.append(np.nan)
            bin_confs.append((lo + hi) / 2)
            bin_counts.append(0)
            continue
        acc = y_true[mask].mean()
        conf = y_prob[mask].mean()
        gap = abs(acc - conf)
        ece += (count / len(y_prob)) * gap
        mce = max(mce, gap)
        bin_accs.append(acc)
        bin_confs.append(conf)
        bin_counts.append(count)
    return ece, mce, bin_edges, bin_accs, bin_confs, bin_counts


def nll(y_true, y_prob):
    eps = 1e-12
    p = np.clip(y_prob, eps, 1 - eps)
    return -np.mean(y_true * np.log(p) + (1 - y_true) * np.log(1 - p))


test_labels_np = test_labels_t.numpy()
probs_before = torch.sigmoid(test_logits).numpy()
probs_after = torch.sigmoid(test_logits / T).numpy()

rows = []
for i, name in enumerate(LABEL_COLS):
    y_true = test_labels_np[:, i]
    if len(np.unique(y_true)) < 2:
        continue

    p_before, p_after = probs_before[:, i], probs_after[:, i]
    ece_b, mce_b, *_ = expected_and_max_calibration_error(y_true, p_before, N_BINS)
    ece_a, mce_a, *_ = expected_and_max_calibration_error(y_true, p_after, N_BINS)

    rows.append({
        "class": name,
        "roc_auc": roc_auc_score(y_true, p_before),
        "pr_auc": average_precision_score(y_true, p_before),
        "ece_before": ece_b, "ece_after": ece_a,
        "mce_before": mce_b, "mce_after": mce_a,
        "brier_before": brier_score_loss(y_true, p_before),
        "brier_after": brier_score_loss(y_true, p_after),
        "nll_before": nll(y_true, p_before),
        "nll_after": nll(y_true, p_after),
    })

calib_results_df = pd.DataFrame(rows).set_index("class")
print("\n" + "=" * 100)
print(f"CALIBRATION METRICS (test split, n={len(test_labels_np)}) — {MODEL_NAME}, T={T:.4f}")
print("=" * 100)
print(calib_results_df.round(4).to_string())

macro_ece_before = calib_results_df["ece_before"].mean()
macro_ece_after = calib_results_df["ece_after"].mean()
macro_brier_before = calib_results_df["brier_before"].mean()
macro_brier_after = calib_results_df["brier_after"].mean()
print(f"\nMacro-avg ECE:   before={macro_ece_before:.4f} -> after={macro_ece_after:.4f}")
print(f"Macro-avg Brier: before={macro_brier_before:.4f} -> after={macro_brier_after:.4f}")

any_idx = LABEL_COLS.index("any")
y_true_any = test_labels_np[:, any_idx]
p_before_any = probs_before[:, any_idx]
p_after_any = probs_after[:, any_idx]

ece_b, mce_b, edges_b, accs_b, confs_b, counts_b = expected_and_max_calibration_error(
    y_true_any, p_before_any, N_BINS)
ece_a, mce_a, edges_a, accs_a, confs_a, counts_a = expected_and_max_calibration_error(
    y_true_any, p_after_any, N_BINS)

fig, axes = plt.subplots(2, 2, figsize=(11, 9))
bin_centers = (edges_b[:-1] + edges_b[1:]) / 2
bin_width = 1 / N_BINS

for col, (accs, counts, ece, mce, label) in enumerate([
    (accs_b, counts_b, ece_b, mce_b, "Before temp scaling (T=1.0)"),
    (accs_a, counts_a, ece_a, mce_a, f"After temp scaling (T={T:.3f})"),
]):
    valid = [not np.isnan(a) for a in accs]
    ax_rel = axes[0, col]
    ax_rel.bar(np.array(bin_centers)[valid], np.array(accs)[valid],
               width=bin_width, edgecolor="black", alpha=0.7, label="Model")
    ax_rel.plot([0, 1], [0, 1], "k--", label="Perfect calibration")
    ax_rel.set_xlabel("Predicted probability")
    ax_rel.set_ylabel("Observed frequency")
    ax_rel.set_title(f"{label}\nECE={ece:.4f}  MCE={mce:.4f}")
    ax_rel.legend(fontsize=8)

    ax_hist = axes[1, col]
    ax_hist.bar(bin_centers, counts, width=bin_width, edgecolor="black", alpha=0.7, color="gray")
    ax_hist.set_xlabel("Predicted probability")
    ax_hist.set_ylabel("Count")
    ax_hist.set_title("Confidence histogram")

plt.suptitle(f"Calibration — {MODEL_NAME}, class 'any' (test, n={len(test_labels_np)})", y=1.00)
plt.tight_layout()
plt.savefig(f"/kaggle/working/{MODEL_NAME}_calibration_reliability.png", dpi=150, bbox_inches="tight")
plt.show()

out_path = f"/kaggle/working/{MODEL_NAME}_calibration_metrics.csv"
calib_results_df.to_csv(out_path)
with open(f"/kaggle/working/{MODEL_NAME}_temperature.txt", "w") as f:
    f.write(f"T={T:.6f}\n")
print(f"\nSaved calibration table to {out_path}")
print(f"Saved fitted T to /kaggle/working/{MODEL_NAME}_temperature.txt")
print(f"Saved reliability diagram to /kaggle/working/{MODEL_NAME}_calibration_reliability.png")

In [ ]:
import shutil
import os
from IPython.display import FileLink

zip_base_name = "/kaggle/working/resmnet"
output_dir = "/kaggle/working/resmnet_outputs"

os.makedirs(output_dir, exist_ok=True)

files_to_zip = [
    f"/kaggle/working/checkpoints/{MODEL_NAME}_best.pt",
    f"/kaggle/working/{MODEL_NAME}_eval_metrics.csv",
    f"/kaggle/working/{MODEL_NAME}_roc_pr_curves.png",
    f"/kaggle/working/{MODEL_NAME}_calibration_metrics.csv",
    f"/kaggle/working/{MODEL_NAME}_calibration_reliability.png",
    f"/kaggle/working/{MODEL_NAME}_temperature.txt",
    "/kaggle/working/test_ids.csv",
]

copied, missing = [], []
for f in files_to_zip:
    if os.path.exists(f):
        shutil.copy(f, output_dir)
        copied.append(f)
    else:
        missing.append(f)

if missing:
    print("Skipped (not found — run the relevant cell first if you need these):")
    for f in missing:
        print(f"  - {f}")

shutil.make_archive(zip_base_name, "zip", output_dir)
print(f"\nZipped {len(copied)} files to {zip_base_name}.zip")

FileLink(f"{zip_base_name}.zip")

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset
from torchvision import models, transforms
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve, f1_score
from tqdm.auto import tqdm

MODEL_NAME = "resnet50"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"

INPUT_DIRS = [
    "/kaggle/input/datasets/anushakirand/rsna-ich-preprocessed-negative-part1",
    "/kaggle/input/datasets/anushakirand/rsna-ich-preprocessed-negative-part2",
    "/kaggle/input/datasets/anushakirand/rsna-ich-preprocessed-negative-part3",
    "/kaggle/input/datasets/anushakirand/rsna-ich-preprocessed-pngs-positive-part-1",
    "/kaggle/input/datasets/anushakirand/rsna-ich-preprocessed-pngs-part3",
    "/kaggle/input/datasets/anushakirand/rsna-ich-preprocessed-pngs",
]
TRAIN_CSV = ("/kaggle/input/competitions/rsna-intracranial-hemorrhage-detection/"
             "rsna-intracranial-hemorrhage-detection/stage_2_train.csv")
MIN_FILE_SIZE_KB = 40
MIN_FILE_SIZE_BYTES = MIN_FILE_SIZE_KB * 1024
VAL_SPLIT = 0.15
TEST_SPLIT = 0.15
LABEL_COLS = ["any", "epidural", "intraparenchymal",
              "intraventricular", "subarachnoid", "subdural"]


class ICHDataset(Dataset):
    def __init__(self, df, id_to_path, transform=None):
        self.df = df
        self.id_to_path = id_to_path
        self.transform = transform
        self.ids = df.index.values
        self.labels = df[LABEL_COLS].values.astype(np.float32)

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        img_id = self.ids[idx]
        path = self.id_to_path[img_id]
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        label = torch.tensor(self.labels[idx], dtype=torch.float32)
        return img, label


def build_resnet50(num_classes=len(LABEL_COLS)):
    model = models.resnet50(weights=None)
    model.fc = nn.Sequential(nn.Dropout(0.3), nn.Linear(model.fc.in_features, num_classes))
    return model.to(DEVICE)


def build_vit_b16(num_classes=len(LABEL_COLS)):
    model = models.vit_b_16(weights=None)
    model.heads = nn.Sequential(nn.Dropout(0.3), nn.Linear(model.hidden_dim, num_classes))
    return model.to(DEVICE)


MODEL_BUILDERS = {"resnet50": build_resnet50, "vit_b16": build_vit_b16}


def load_labels(csv_path):
    y = pd.read_csv(csv_path)
    id_split = y.ID.str.rsplit("_", n=1, expand=True)
    y = pd.concat([id_split, y.Label], axis=1)
    y.columns = ["id", "sub_type", "label"]
    y = y.drop_duplicates(subset=["id", "sub_type"])
    return y.pivot(index="id", columns="sub_type", values="label")


def scan_input_dirs(input_dirs, min_size_bytes):
    id_to_path = {}
    skipped_small = 0
    for d in input_dirs:
        if not os.path.isdir(d):
            print(f"WARNING: directory not found, skipping: {d}")
            continue
        filenames = [f for f in os.listdir(d) if f.endswith(".png")]
        for f in tqdm(filenames, desc=f"scanning {os.path.basename(d)}"):
            full_path = os.path.join(d, f)
            if os.path.getsize(full_path) <= min_size_bytes:
                skipped_small += 1
                continue
            id_to_path[f[:-4]] = full_path
    print(f"Scanned {len(input_dirs)} directories: {len(id_to_path)} usable PNGs "
          f"found, {skipped_small} filtered out as low-information (<= "
          f"{min_size_bytes} bytes / {MIN_FILE_SIZE_KB}KB).")
    return id_to_path


def make_splits(df):
    train_df, temp_df = train_test_split(
        df, test_size=VAL_SPLIT + TEST_SPLIT, stratify=df["any"], random_state=42
    )
    val_df, test_df = train_test_split(
        temp_df, test_size=TEST_SPLIT / (VAL_SPLIT + TEST_SPLIT),
        stratify=temp_df["any"], random_state=42
    )
    return train_df, val_df, test_df


print("Scanning input directories (same as training run)...")
id_to_path = scan_input_dirs(INPUT_DIRS, MIN_FILE_SIZE_BYTES)
df = load_labels(TRAIN_CSV)
df = df[df.index.isin(id_to_path.keys())]
train_df, val_df, test_df = make_splits(df)
print(f"Rebuilt splits — train: {len(train_df)}, val: {len(val_df)}, test: {len(test_df)}")

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

model = MODEL_BUILDERS[MODEL_NAME]()
checkpoint = torch.load(f"/kaggle/working/checkpoints/{MODEL_NAME}_best.pt",
                         map_location=DEVICE, weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()
print(f"Loaded checkpoint: epoch {checkpoint['epoch']}, val_loss={checkpoint['val_loss']:.4f}")

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score, average_precision_score, precision_recall_curve,
    confusion_matrix, f1_score,
)
from tqdm.auto import tqdm

# ------------------------------------------------------------------
# CONFIG (same as training run)
# ------------------------------------------------------------------
MODEL_NAME = "resnet50"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
print(f"Using device: {DEVICE}")

INPUT_DIRS = [
    "/kaggle/input/datasets/anushakirand/rsna-ich-preprocessed-negative-part1",
    "/kaggle/input/datasets/anushakirand/rsna-ich-preprocessed-negative-part2",
    "/kaggle/input/datasets/anushakirand/rsna-ich-preprocessed-negative-part3",
    "/kaggle/input/datasets/anushakirand/rsna-ich-preprocessed-pngs-positive-part-1",
    "/kaggle/input/datasets/anushakirand/rsna-ich-preprocessed-pngs-part3",
    "/kaggle/input/datasets/anushakirand/rsna-ich-preprocessed-pngs",
]
TRAIN_CSV = ("/kaggle/input/competitions/rsna-intracranial-hemorrhage-detection/"
             "rsna-intracranial-hemorrhage-detection/stage_2_train.csv")
MIN_FILE_SIZE_KB = 40
MIN_FILE_SIZE_BYTES = MIN_FILE_SIZE_KB * 1024
VAL_SPLIT = 0.15
TEST_SPLIT = 0.15
LABEL_COLS = ["any", "epidural", "intraparenchymal",
              "intraventricular", "subarachnoid", "subdural"]
CHECKPOINT_PATH = f"/kaggle/working/checkpoints/{MODEL_NAME}_best.pt"


class ICHDataset(Dataset):
    def __init__(self, df, id_to_path, transform=None):
        self.df = df
        self.id_to_path = id_to_path
        self.transform = transform
        self.ids = df.index.values
        self.labels = df[LABEL_COLS].values.astype(np.float32)

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        img_id = self.ids[idx]
        path = self.id_to_path[img_id]
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        label = torch.tensor(self.labels[idx], dtype=torch.float32)
        return img, label


def build_resnet50(num_classes=len(LABEL_COLS)):
    model = models.resnet50(weights=None)
    model.fc = nn.Sequential(nn.Dropout(0.3), nn.Linear(model.fc.in_features, num_classes))
    return model.to(DEVICE)


def build_vit_b16(num_classes=len(LABEL_COLS)):
    model = models.vit_b_16(weights=None)
    model.heads = nn.Sequential(nn.Dropout(0.3), nn.Linear(model.hidden_dim, num_classes))
    return model.to(DEVICE)


MODEL_BUILDERS = {"resnet50": build_resnet50, "vit_b16": build_vit_b16}


def load_labels(csv_path):
    y = pd.read_csv(csv_path)
    id_split = y.ID.str.rsplit("_", n=1, expand=True)
    y = pd.concat([id_split, y.Label], axis=1)
    y.columns = ["id", "sub_type", "label"]
    y = y.drop_duplicates(subset=["id", "sub_type"])
    return y.pivot(index="id", columns="sub_type", values="label")


def scan_input_dirs(input_dirs, min_size_bytes):
    id_to_path = {}
    skipped_small = 0
    for d in input_dirs:
        if not os.path.isdir(d):
            print(f"WARNING: directory not found, skipping: {d}")
            continue
        filenames = [f for f in os.listdir(d) if f.endswith(".png")]
        for f in tqdm(filenames, desc=f"scanning {os.path.basename(d)}"):
            full_path = os.path.join(d, f)
            if os.path.getsize(full_path) <= min_size_bytes:
                skipped_small += 1
                continue
            id_to_path[f[:-4]] = full_path
    print(f"Scanned {len(input_dirs)} directories: {len(id_to_path)} usable PNGs "
          f"found, {skipped_small} filtered out.")
    return id_to_path


def make_splits(df):
    train_df, temp_df = train_test_split(
        df, test_size=VAL_SPLIT + TEST_SPLIT, stratify=df["any"], random_state=42
    )
    val_df, test_df = train_test_split(
        temp_df, test_size=TEST_SPLIT / (VAL_SPLIT + TEST_SPLIT),
        stratify=temp_df["any"], random_state=42
    )
    return train_df, val_df, test_df


# ------------------------------------------------------------------
# REBUILD: data + model
# ------------------------------------------------------------------
print("Scanning input directories...")
id_to_path = scan_input_dirs(INPUT_DIRS, MIN_FILE_SIZE_BYTES)
df = load_labels(TRAIN_CSV)
df = df[df.index.isin(id_to_path.keys())]
train_df, val_df, test_df = make_splits(df)
print(f"Rebuilt splits — train: {len(train_df)}, val: {len(val_df)}, test: {len(test_df)}")

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
val_ds = ICHDataset(val_df, id_to_path, transform=eval_transform)
test_ds = ICHDataset(test_df, id_to_path, transform=eval_transform)

val_loader = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

model = MODEL_BUILDERS[MODEL_NAME]()
checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()
print(f"Loaded checkpoint: epoch {checkpoint['epoch']}, val_loss={checkpoint['val_loss']:.4f}")


# ------------------------------------------------------------------
# INFERENCE (with progress bars this time)
# ------------------------------------------------------------------
@torch.no_grad()
def run_inference(loader, desc="inference"):
    all_probs, all_labels = [], []
    for imgs, labels in tqdm(loader, desc=desc):
        imgs = imgs.to(DEVICE)
        with torch.autocast(device_type=DEVICE.type, enabled=USE_AMP):
            outputs = model(imgs)
        probs = torch.sigmoid(outputs.float()).cpu().numpy()
        all_probs.append(probs)
        all_labels.append(labels.numpy())
    return np.concatenate(all_probs), np.concatenate(all_labels)


val_probs, val_labels = run_inference(val_loader, desc="val inference")
test_probs, test_labels = run_inference(test_loader, desc="test inference")
print(f"Done: {len(val_probs)} val images, {len(test_probs)} test images.")

# Cache raw arrays to disk immediately — so if the session stops after
# this, next time you can np.load() this instead of rerunning inference.
np.savez("/kaggle/working/test_inference_cache.npz",
         val_probs=val_probs, val_labels=val_labels,
         test_probs=test_probs, test_labels=test_labels)
print("Cached val/test probs+labels to /kaggle/working/test_inference_cache.npz")


# ------------------------------------------------------------------
# METRICS TABLE
# ------------------------------------------------------------------
def best_f1_threshold(y_true, y_prob):
    precisions, recalls, thresholds = precision_recall_curve(y_true, y_prob)
    f1s = 2 * precisions * recalls / (precisions + recalls + 1e-12)
    best_idx = np.nanargmax(f1s[:-1])
    return thresholds[best_idx] if len(thresholds) > 0 else 0.5


rows = []
confusion_matrices = {}

for i, name in enumerate(LABEL_COLS):
    y_true_val = val_labels[:, i]
    y_true_test = test_labels[:, i]
    y_prob_test = test_probs[:, i]

    if len(np.unique(y_true_val)) < 2 or len(np.unique(y_true_test)) < 2:
        continue

    thresh = best_f1_threshold(y_true_val, val_probs[:, i])
    y_pred_test = (y_prob_test >= thresh).astype(int)

    roc_auc = roc_auc_score(y_true_test, y_prob_test)
    pr_auc = average_precision_score(y_true_test, y_prob_test)

    tn, fp, fn, tp = confusion_matrix(y_true_test, y_pred_test, labels=[0, 1]).ravel()
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    f1 = f1_score(y_true_test, y_pred_test, zero_division=0)

    confusion_matrices[name] = np.array([[tn, fp], [fn, tp]])

    rows.append({
        "class": name, "roc_auc": roc_auc, "pr_auc": pr_auc,
        "threshold_from_val": thresh, "precision": precision,
        "recall_sensitivity": recall, "specificity": specificity, "f1": f1,
        "n_positive": int(y_true_test.sum()), "n_total": len(y_true_test),
    })

results_df = pd.DataFrame(rows).set_index("class")
macro_auc = roc_auc_score(test_labels, test_probs, average="macro")
micro_auc = roc_auc_score(test_labels, test_probs, average="micro")

print("\n" + "=" * 90)
print(f"PER-CLASS METRICS (test split) — {MODEL_NAME}, checkpoint epoch {checkpoint['epoch']}")
print("=" * 90)
print(results_df.round(4).to_string())
print(f"\nMacro-average ROC-AUC: {macro_auc:.4f}")
print(f"Micro-average ROC-AUC: {micro_auc:.4f}")

out_path = f"/kaggle/working/{MODEL_NAME}_eval_metrics.csv"
results_df.to_csv(out_path)
print(f"\nFull metrics table saved to {out_path}")

In [ ]:
import shutil
zip_base_name = "/kaggle/working/resmnet_eval_backup"
output_dir = "/kaggle/working/resmnet_eval_backup_files"
import os
os.makedirs(output_dir, exist_ok=True)

for f in [f"/kaggle/working/{MODEL_NAME}_eval_metrics.csv",
          "/kaggle/working/test_inference_cache.npz"]:
    if os.path.exists(f):
        shutil.copy(f, output_dir)

shutil.make_archive(zip_base_name, "zip", output_dir)
print(f"Backed up to {zip_base_name}.zip")

from IPython.display import FileLink
FileLink(f"{zip_base_name}.zip")

In [ ]:
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
import os

MODEL_NAME = "resnet50"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
LABEL_COLS = ["any", "epidural", "intraparenchymal",
              "intraventricular", "subarachnoid", "subdural"]

# ------------------------------------------------------------------
# Load everything that's already saved — no inference rerun
# ------------------------------------------------------------------
results_df = pd.read_csv(f"/kaggle/working/{MODEL_NAME}_eval_metrics.csv", index_col="class")
cache = np.load("/kaggle/working/test_inference_cache.npz")
val_probs, val_labels = cache["val_probs"], cache["val_labels"]
test_probs, test_labels = cache["test_probs"], cache["test_labels"]
print(f"Loaded cached results_df and inference arrays "
      f"(val: {len(val_probs)}, test: {len(test_probs)})")

# Rebuild test_df from the saved id list + original labels CSV (fast, no image scan)
TRAIN_CSV = ("/kaggle/input/competitions/rsna-intracranial-hemorrhage-detection/"
             "rsna-intracranial-hemorrhage-detection/stage_2_train.csv")

def load_labels(csv_path):
    y = pd.read_csv(csv_path)
    id_split = y.ID.str.rsplit("_", n=1, expand=True)
    y = pd.concat([id_split, y.Label], axis=1)
    y.columns = ["id", "sub_type", "label"]
    y = y.drop_duplicates(subset=["id", "sub_type"])
    return y.pivot(index="id", columns="sub_type", values="label")

test_ids = pd.read_csv("/kaggle/working/test_ids.csv")["id"].values
full_labels = load_labels(TRAIN_CSV)
test_df = full_labels.loc[test_ids]
print(f"Rebuilt test_df from test_ids.csv: {len(test_df)} rows — no image scan needed for this part.")

# ------------------------------------------------------------------
# id_to_path — this part CANNOT be skipped without a prior cache.
# Only scanning for ids that appear in test_df (not the full ~270K),
# which is faster than a full training-time scan but still requires
# listing each directory once.
# ------------------------------------------------------------------
INPUT_DIRS = [
    "/kaggle/input/datasets/anushakirand/rsna-ich-preprocessed-negative-part1",
    "/kaggle/input/datasets/anushakirand/rsna-ich-preprocessed-negative-part2",
    "/kaggle/input/datasets/anushakirand/rsna-ich-preprocessed-negative-part3",
    "/kaggle/input/datasets/anushakirand/rsna-ich-preprocessed-pngs-positive-part-1",
    "/kaggle/input/datasets/anushakirand/rsna-ich-preprocessed-pngs-part3",
    "/kaggle/input/datasets/anushakirand/rsna-ich-preprocessed-pngs",
]
needed_ids = set(test_df.index)
id_to_path = {}
for d in INPUT_DIRS:
    if not os.path.isdir(d):
        continue
    for f in os.listdir(d):
        if f.endswith(".png"):
            img_id = f[:-4]
            if img_id in needed_ids:
                id_to_path[img_id] = os.path.join(d, f)

print(f"Resolved paths for {len(id_to_path)} / {len(needed_ids)} test images.")

# Cache this mapping too, so next time even this scan is skippable
with open("/kaggle/working/test_id_to_path.json", "w") as f:
    json.dump(id_to_path, f)
print("Cached id_to_path to test_id_to_path.json for future sessions.")

def build_resnet50(num_classes=len(LABEL_COLS)):
    model = models.resnet50(weights=None)
    model.fc = nn.Sequential(nn.Dropout(0.3), nn.Linear(model.fc.in_features, num_classes))
    return model.to(DEVICE)

MODEL_BUILDERS = {"resnet50": build_resnet50}
model = MODEL_BUILDERS[MODEL_NAME]()
checkpoint = torch.load(f"/kaggle/working/checkpoints/{MODEL_NAME}_best.pt",
                         map_location=DEVICE, weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
print("Model loaded and ready. Grad-CAM cell can now run directly.")

In [ ]:
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
import os

MODEL_NAME = "resnet50"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
LABEL_COLS = ["any", "epidural", "intraparenchymal",
              "intraventricular", "subarachnoid", "subdural"]

# ------------------------------------------------------------------
# Load everything that's already saved — no inference rerun
# ------------------------------------------------------------------
results_df = pd.read_csv(f"/kaggle/working/{MODEL_NAME}_eval_metrics.csv", index_col="class")
cache = np.load("/kaggle/working/test_inference_cache.npz")
val_probs, val_labels = cache["val_probs"], cache["val_labels"]
test_probs, test_labels = cache["test_probs"], cache["test_labels"]
print(f"Loaded cached results_df and inference arrays "
      f"(val: {len(val_probs)}, test: {len(test_probs)})")

# Rebuild test_df from the saved id list + original labels CSV (fast, no image scan)
TRAIN_CSV = ("/kaggle/input/competitions/rsna-intracranial-hemorrhage-detection/"
             "rsna-intracranial-hemorrhage-detection/stage_2_train.csv")

def load_labels(csv_path):
    y = pd.read_csv(csv_path)
    id_split = y.ID.str.rsplit("_", n=1, expand=True)
    y = pd.concat([id_split, y.Label], axis=1)
    y.columns = ["id", "sub_type", "label"]
    y = y.drop_duplicates(subset=["id", "sub_type"])
    return y.pivot(index="id", columns="sub_type", values="label")

test_ids = pd.read_csv("/kaggle/working/test_ids.csv")["id"].values
full_labels = load_labels(TRAIN_CSV)
test_df = full_labels.loc[test_ids]
print(f"Rebuilt test_df from test_ids.csv: {len(test_df)} rows — no image scan needed for this part.")

# ------------------------------------------------------------------
# id_to_path — this part CANNOT be skipped without a prior cache.
# Only scanning for ids that appear in test_df (not the full ~270K),
# which is faster than a full training-time scan but still requires
# listing each directory once.
# ------------------------------------------------------------------
INPUT_DIRS = [
    "/kaggle/input/datasets/anushakirand/rsna-ich-preprocessed-negative-part1",
    "/kaggle/input/datasets/anushakirand/rsna-ich-preprocessed-negative-part2",
    "/kaggle/input/datasets/anushakirand/rsna-ich-preprocessed-negative-part3",
    "/kaggle/input/datasets/anushakirand/rsna-ich-preprocessed-pngs-positive-part-1",
    "/kaggle/input/datasets/anushakirand/rsna-ich-preprocessed-pngs-part3",
    "/kaggle/input/datasets/anushakirand/rsna-ich-preprocessed-pngs",
]
needed_ids = set(test_df.index)
id_to_path = {}
for d in INPUT_DIRS:
    if not os.path.isdir(d):
        continue
    for f in os.listdir(d):
        if f.endswith(".png"):
            img_id = f[:-4]
            if img_id in needed_ids:
                id_to_path[img_id] = os.path.join(d, f)

print(f"Resolved paths for {len(id_to_path)} / {len(needed_ids)} test images.")

# Cache this mapping too, so next time even this scan is skippable
with open("/kaggle/working/test_id_to_path.json", "w") as f:
    json.dump(id_to_path, f)
print("Cached id_to_path to test_id_to_path.json for future sessions.")

# ------------------------------------------------------------------
# Model + eval_transform
# ------------------------------------------------------------------
def build_resnet50(num_classes=len(LABEL_COLS)):
    model = models.resnet50(weights=None)
    model.fc = nn.Sequential(nn.Dropout(0.3), nn.Linear(model.fc.in_features, num_classes))
    return model.to(DEVICE)

MODEL_BUILDERS = {"resnet50": build_resnet50}
model = MODEL_BUILDERS[MODEL_NAME]()
checkpoint = torch.load(f"/kaggle/working/checkpoints/{MODEL_NAME}_best.pt",
                         map_location=DEVICE, weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
print("Model loaded and ready. Grad-CAM cell can now run directly.")

In [ ]:
for name in ["model", "test_df", "id_to_path", "eval_transform",
             "results_df", "test_probs", "test_labels", "MODEL_BUILDERS", "LABEL_COLS"]:
    print(f"{name}: {'present' if name in globals() else 'MISSING'}")

In [ ]:
import zipfile, os

SKIP_EXT = {".zip"}
SKIP_DIRS = {"checkpoints"}  # drop this if you actually want the model weights included

with zipfile.ZipFile("/kaggle/working/all_outputs.zip", "w", zipfile.ZIP_STORED) as zf:
    for root, dirs, files in os.walk("/kaggle/working"):
        dirs[:] = [d for d in dirs if d not in SKIP_DIRS]
        for f in files:
            if os.path.splitext(f)[1] in SKIP_EXT:
                continue
            full = os.path.join(root, f)
            zf.write(full, os.path.relpath(full, "/kaggle/working"))

print("Done")

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Sampler
from torchvision import models, transforms
from PIL import Image
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

# ------------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------------
MODEL_NAME = "resnet50"   # "resnet50" or "vit_b16"

INPUT_DIRS = [
    "/kaggle/input/datasets/anushakirand/rsna-ich-preprocessed-negative-part1",
    "/kaggle/input/datasets/anushakirand/rsna-ich-preprocessed-negative-part2",
    "/kaggle/input/datasets/anushakirand/rsna-ich-preprocessed-negative-part3",
    "/kaggle/input/datasets/anushakirand/rsna-ich-preprocessed-pngs-positive-part-1",
    "/kaggle/input/datasets/anushakirand/rsna-ich-preprocessed-pngs-part3",
    "/kaggle/input/datasets/anushakirand/rsna-ich-preprocessed-pngs",
]

TRAIN_CSV = ("/kaggle/input/competitions/rsna-intracranial-hemorrhage-detection/"
             "rsna-intracranial-hemorrhage-detection/stage_2_train.csv")

MIN_FILE_SIZE_KB = 40
MIN_FILE_SIZE_BYTES = MIN_FILE_SIZE_KB * 1024

LABEL_COLS = ["any", "epidural", "intraparenchymal",
              "intraventricular", "subarachnoid", "subdural"]

BATCH_SIZE = 32
NUM_EPOCHS = 15
LR = 1e-4

# Three-way split. 70/15/15 by default — change if you want 70/10/20 etc.
VAL_SPLIT = 0.15
TEST_SPLIT = 0.15

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"

EARLY_STOPPING_PATIENCE = 5
CHECKPOINT_DIR = "/kaggle/working/checkpoints"
TEST_IDS_PATH = "/kaggle/working/test_ids.csv"

NUM_WORKERS = 4


def load_labels(csv_path):
    y = pd.read_csv(csv_path)
    id_split = y.ID.str.rsplit("_", n=1, expand=True)
    y = pd.concat([id_split, y.Label], axis=1)
    y.columns = ["id", "sub_type", "label"]
    y = y.drop_duplicates(subset=["id", "sub_type"])
    df = y.pivot(index="id", columns="sub_type", values="label")
    return df


class ICHDataset(Dataset):
    def __init__(self, df, id_to_path, transform=None):
        self.df = df
        self.id_to_path = id_to_path
        self.transform = transform
        self.ids = df.index.values
        self.labels = df[LABEL_COLS].values.astype(np.float32)

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        img_id = self.ids[idx]
        path = self.id_to_path[img_id]
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        label = torch.tensor(self.labels[idx], dtype=torch.float32)
        return img, label


class BalancedRandomSampler(Sampler):
    def __init__(self, labels_any):
        self.pos_idx = np.where(labels_any == 1)[0]
        self.neg_idx = np.where(labels_any == 0)[0]

    def __iter__(self):
        n = min(len(self.neg_idx), max(len(self.pos_idx), 1))
        neg_sample = np.random.choice(self.neg_idx, n, replace=False)
        pos_sample = (np.random.choice(self.pos_idx, n, replace=False)
                      if len(self.pos_idx) > 0 else np.array([], dtype=int))
        ids = np.concatenate([pos_sample, neg_sample])
        np.random.shuffle(ids)
        return iter(ids.tolist())

    def __len__(self):
        return min(len(self.neg_idx), max(len(self.pos_idx), 1)) * 2


def build_resnet50(num_classes=len(LABEL_COLS)):
    model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
    for param in model.parameters():
        param.requires_grad = False
    for param in model.layer4.parameters():
        param.requires_grad = True
    model.fc = nn.Sequential(
        nn.Dropout(0.3),
        nn.Linear(model.fc.in_features, num_classes),
    )
    return model.to(DEVICE)


def build_vit_b16(num_classes=len(LABEL_COLS)):
    model = models.vit_b_16(weights=models.ViT_B_16_Weights.IMAGENET1K_V1)
    for param in model.parameters():
        param.requires_grad = False
    for param in model.encoder.layers[-1].parameters():
        param.requires_grad = True
    model.heads = nn.Sequential(
        nn.Dropout(0.3),
        nn.Linear(model.hidden_dim, num_classes),
    )
    for param in model.heads.parameters():
        param.requires_grad = True
    return model.to(DEVICE)


MODEL_BUILDERS = {"resnet50": build_resnet50, "vit_b16": build_vit_b16}

OPTIMIZER_CONFIG = {
    "resnet50": {"type": "sgd", "lr": 1e-3, "momentum": 0.9},
    "vit_b16": {"type": "adamw", "lr": 1e-4, "weight_decay": 1e-4},
}

SCHEDULER_CONFIG = {
    "resnet50": {"warmup_epochs": 0},
    "vit_b16": {"warmup_epochs": 1},
}


def build_optimizer(model, model_name):
    cfg = OPTIMIZER_CONFIG[model_name]
    trainable_params = filter(lambda p: p.requires_grad, model.parameters())
    if cfg["type"] == "sgd":
        return torch.optim.SGD(trainable_params, lr=cfg["lr"], momentum=cfg["momentum"])
    elif cfg["type"] == "adamw":
        return torch.optim.AdamW(trainable_params, lr=cfg["lr"],
                                  weight_decay=cfg.get("weight_decay", 0.0))
    else:
        raise ValueError(f"Unknown optimizer type: {cfg['type']}")


def build_scheduler(optimizer, model_name, num_epochs):
    warmup_epochs = SCHEDULER_CONFIG[model_name]["warmup_epochs"]
    warmup_epochs = min(warmup_epochs, max(num_epochs - 1, 0))

    if warmup_epochs > 0:
        warmup = torch.optim.lr_scheduler.LinearLR(
            optimizer, start_factor=0.1, end_factor=1.0, total_iters=warmup_epochs
        )
        cosine = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=max(num_epochs - warmup_epochs, 1)
        )
        scheduler = torch.optim.lr_scheduler.SequentialLR(
            optimizer, schedulers=[warmup, cosine], milestones=[warmup_epochs]
        )
    else:
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)

    return scheduler


def train_one_epoch(model, loader, optimizer, criterion, scaler, epoch, num_epochs):
    model.train()
    total_loss = 0.0
    pbar = tqdm(loader, desc=f"train epoch {epoch+1}/{num_epochs}", leave=False)
    for imgs, labels in pbar:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()

        with torch.autocast(device_type=DEVICE.type, enabled=USE_AMP):
            outputs = model(imgs)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * imgs.size(0)
        pbar.set_postfix(loss=f"{loss.item():.4f}")
    return total_loss / len(loader.dataset)


@torch.no_grad()
def evaluate(model, loader, criterion, epoch, num_epochs):
    model.eval()
    total_loss = 0.0
    all_preds, all_labels = [], []
    pbar = tqdm(loader, desc=f"val epoch {epoch+1}/{num_epochs}", leave=False)
    for imgs, labels in pbar:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)

        with torch.autocast(device_type=DEVICE.type, enabled=USE_AMP):
            outputs = model(imgs)
            loss = criterion(outputs, labels)

        total_loss += loss.item() * imgs.size(0)
        all_preds.append(torch.sigmoid(outputs.float()).cpu().numpy())
        all_labels.append(labels.cpu().numpy())

    all_preds = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)

    aucs = {}
    for i, name in enumerate(LABEL_COLS):
        if len(np.unique(all_labels[:, i])) > 1:
            aucs[name] = roc_auc_score(all_labels[:, i], all_preds[:, i])
        else:
            aucs[name] = float("nan")

    return total_loss / len(loader.dataset), aucs


def run_training(model_name, train_ds, val_ds, y_train):
    print(f"\n{'='*60}")
    print(f"FULL TRAINING RUN: {model_name}")
    print(f"{'='*60}")

    model = MODEL_BUILDERS[model_name]()
    criterion = nn.BCEWithLogitsLoss()
    optimizer = build_optimizer(model, model_name)
    scheduler = build_scheduler(optimizer, model_name, NUM_EPOCHS)
    scaler = torch.amp.GradScaler(device=DEVICE.type, enabled=USE_AMP)

    sampler = BalancedRandomSampler(y_train["any"].values)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                               num_workers=NUM_WORKERS, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                             num_workers=NUM_WORKERS, pin_memory=True)

    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    checkpoint_path = os.path.join(CHECKPOINT_DIR, f"{model_name}_best.pt")

    best_val_loss = float("inf")
    epochs_no_improve = 0

    for epoch in range(NUM_EPOCHS):
        current_lr = optimizer.param_groups[0]["lr"]

        train_loss = train_one_epoch(model, train_loader, optimizer, criterion,
                                      scaler, epoch, NUM_EPOCHS)
        val_loss, val_aucs = evaluate(model, val_loader, criterion, epoch, NUM_EPOCHS)

        auc_str = ", ".join(f"{k}={v:.3f}" for k, v in val_aucs.items())
        print(f"[{model_name}] Epoch {epoch+1}/{NUM_EPOCHS} | lr={current_lr:.2e} | "
              f"train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | {auc_str}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            epochs_no_improve = 0
            torch.save({
                "model_name": model_name,
                "epoch": epoch + 1,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "val_loss": val_loss,
                "val_aucs": val_aucs,
            }, checkpoint_path)
            print(f"[{model_name}]   -> val loss improved, checkpoint saved to {checkpoint_path}")
        else:
            epochs_no_improve += 1
            print(f"[{model_name}]   -> no improvement ({epochs_no_improve}/{EARLY_STOPPING_PATIENCE})")

        scheduler.step()

        if epochs_no_improve >= EARLY_STOPPING_PATIENCE:
            print(f"[{model_name}] Early stopping triggered at epoch {epoch+1}.")
            break

    print(f"[{model_name}] Training complete. Best val_loss={best_val_loss:.4f}, "
          f"checkpoint at {checkpoint_path}")


def scan_input_dirs(input_dirs, min_size_bytes):
    id_to_path = {}
    skipped_small = 0
    for d in input_dirs:
        if not os.path.isdir(d):
            print(f"WARNING: directory not found, skipping: {d}")
            continue
        filenames = [f for f in os.listdir(d) if f.endswith(".png")]
        for f in tqdm(filenames, desc=f"scanning {os.path.basename(d)}"):
            full_path = os.path.join(d, f)
            if os.path.getsize(full_path) <= min_size_bytes:
                skipped_small += 1
                continue
            img_id = f[:-4]
            id_to_path[img_id] = full_path
    print(f"Scanned {len(input_dirs)} directories: {len(id_to_path)} usable PNGs "
          f"found, {skipped_small} filtered out as low-information (<= "
          f"{min_size_bytes} bytes / {MIN_FILE_SIZE_KB}KB).")
    return id_to_path


def make_splits(df):
    train_df, temp_df = train_test_split(
        df, test_size=VAL_SPLIT + TEST_SPLIT, stratify=df["any"], random_state=42
    )
    val_df, test_df = train_test_split(
        temp_df, test_size=TEST_SPLIT / (VAL_SPLIT + TEST_SPLIT),
        stratify=temp_df["any"], random_state=42
    )
    return train_df, val_df, test_df


def main():
    print(f"*** FULL TRAINING RUN: {MODEL_NAME} ***")
    print(f"Using device: {DEVICE} | AMP enabled: {USE_AMP}")

    id_to_path = scan_input_dirs(INPUT_DIRS, MIN_FILE_SIZE_BYTES)

    if len(id_to_path) == 0:
        print("\nNo usable PNGs found across INPUT_DIRS. Check the paths are "
              "correct and the datasets are actually attached as Input.")
        return

    df = load_labels(TRAIN_CSV)
    df = df[df.index.isin(id_to_path.keys())]
    print(f"Matched {len(df)} of those to labels. "
          f"Positive rate: {df['any'].mean():.4f}")

    train_df, val_df, test_df = make_splits(df)
    print(f"Full run sizes — train: {len(train_df)}, val: {len(val_df)}, "
          f"test: {len(test_df)}")

    os.makedirs(os.path.dirname(TEST_IDS_PATH), exist_ok=True)
    test_df.index.to_series(name="id").to_csv(TEST_IDS_PATH, index=False)
    print(f"Saved test set ids to {TEST_IDS_PATH}")

    train_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                              std=[0.229, 0.224, 0.225]),
    ])
    eval_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                              std=[0.229, 0.224, 0.225]),
    ])

    global train_df_g, val_df_g, test_df_g, id_to_path_g
    train_df_g, val_df_g, test_df_g, id_to_path_g = train_df, val_df, test_df, id_to_path

    train_ds = ICHDataset(train_df, id_to_path, transform=train_transform)
    val_ds = ICHDataset(val_df, id_to_path, transform=eval_transform)

    global val_ds_g
    val_ds_g = val_ds

    run_training(MODEL_NAME, train_ds, val_ds, train_df)


main()

In [ ]:
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
import os

MODEL_NAME = "resnet50"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
LABEL_COLS = ["any", "epidural", "intraparenchymal",
              "intraventricular", "subarachnoid", "subdural"]
print(f"Using device: {DEVICE}")

# ------------------------------------------------------------------
# Load everything already saved — no inference rerun, no full scan
# ------------------------------------------------------------------
results_df = pd.read_csv(f"/kaggle/working/{MODEL_NAME}_eval_metrics.csv", index_col="class")
cache = np.load("/kaggle/working/test_inference_cache.npz")
val_probs, val_labels = cache["val_probs"], cache["val_labels"]
test_probs, test_labels = cache["test_probs"], cache["test_labels"]
print(f"Loaded cached results_df and inference arrays "
      f"(val: {len(val_probs)}, test: {len(test_probs)})")

TRAIN_CSV = ("/kaggle/input/competitions/rsna-intracranial-hemorrhage-detection/"
             "rsna-intracranial-hemorrhage-detection/stage_2_train.csv")

def load_labels(csv_path):
    y = pd.read_csv(csv_path)
    id_split = y.ID.str.rsplit("_", n=1, expand=True)
    y = pd.concat([id_split, y.Label], axis=1)
    y.columns = ["id", "sub_type", "label"]
    y = y.drop_duplicates(subset=["id", "sub_type"])
    return y.pivot(index="id", columns="sub_type", values="label")

test_ids = pd.read_csv("/kaggle/working/test_ids.csv")["id"].values
full_labels = load_labels(TRAIN_CSV)
test_df = full_labels.loc[test_ids]
print(f"Rebuilt test_df from test_ids.csv: {len(test_df)} rows")

# id_to_path — load from cache if it exists, else scan (only for test ids)
cache_path = "/kaggle/working/test_id_to_path.json"
if os.path.exists(cache_path):
    print("Loading cached id_to_path from disk — no scan needed.")
    with open(cache_path) as f:
        id_to_path = json.load(f)
else:
    print("No cached id_to_path found — scanning (test ids only)...")
    INPUT_DIRS = [
        "/kaggle/input/datasets/anushakirand/rsna-ich-preprocessed-negative-part1",
        "/kaggle/input/datasets/anushakirand/rsna-ich-preprocessed-negative-part2",
        "/kaggle/input/datasets/anushakirand/rsna-ich-preprocessed-negative-part3",
        "/kaggle/input/datasets/anushakirand/rsna-ich-preprocessed-pngs-positive-part-1",
        "/kaggle/input/datasets/anushakirand/rsna-ich-preprocessed-pngs-part3",
        "/kaggle/input/datasets/anushakirand/rsna-ich-preprocessed-pngs",
    ]
    needed_ids = set(test_df.index)
    id_to_path = {}
    for d in INPUT_DIRS:
        if not os.path.isdir(d):
            continue
        for f in os.listdir(d):
            if f.endswith(".png"):
                img_id = f[:-4]
                if img_id in needed_ids:
                    id_to_path[img_id] = os.path.join(d, f)
    with open(cache_path, "w") as f:
        json.dump(id_to_path, f)
print(f"id_to_path ready: {len(id_to_path)} images resolved.")

# Model
def build_resnet50(num_classes=len(LABEL_COLS)):
    model = models.resnet50(weights=None)
    model.fc = nn.Sequential(nn.Dropout(0.3), nn.Linear(model.fc.in_features, num_classes))
    return model.to(DEVICE)

MODEL_BUILDERS = {"resnet50": build_resnet50}
model = MODEL_BUILDERS[MODEL_NAME]()
checkpoint = torch.load(f"/kaggle/working/checkpoints/{MODEL_NAME}_best.pt",
                         map_location=DEVICE, weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()
print(f"Loaded checkpoint: epoch {checkpoint['epoch']}, val_loss={checkpoint['val_loss']:.4f}")

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
print("Ready — run the Grad-CAM cell next.")

In [ ]:
!pip install grad-cam -q

import os
import json
import pickle
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image

# ------------------------------------------------------------------
# GRAD-CAM — scaled up sample, for stable AOPC/Max-Sensitivity input.
# Computes+caches CAMs for N_CAM_SAMPLES confirmed TPs per class,
# but only visualizes a small subset (grid would be unusable at N=30).
# ------------------------------------------------------------------
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

N_CAM_SAMPLES = 30          # per class, capped by availability (esp. epidural)
N_VISUALIZE_PER_CLASS = 2   # kept small — just a sanity-check grid

GRADCAM_OUT_DIR = "/kaggle/working/gradcam_examples"
CAM_CACHE_PATH = "/kaggle/working/cam_cache.pkl"
os.makedirs(GRADCAM_OUT_DIR, exist_ok=True)

CHANNEL_NAMES = ["brain", "subdural", "bone"]

target_layers = [model.layer4[-1]]
cam = GradCAM(model=model, target_layers=target_layers)

cam_cache = {}          # img_id -> {class_name: grayscale_cam}
img_cache = {}          # img_id -> img_np (3-window array), reused by AOPC/Max-Sensitivity
bone_corr_records = []


def load_image_for_cam(path):
    img = Image.open(path).convert("RGB").resize((224, 224))
    img_np = np.array(img).astype(np.float32) / 255.0
    input_tensor = eval_transform(img).unsqueeze(0).to(DEVICE)
    return input_tensor, img_np


def bone_shortcut_correlation(grayscale_cam, bone_channel):
    cam_flat = grayscale_cam.flatten()
    bone_flat = bone_channel.flatten()
    if cam_flat.std() < 1e-8 or bone_flat.std() < 1e-8:
        return np.nan
    return float(np.corrcoef(cam_flat, bone_flat)[0, 1])


def compute_gradcam(img_id, class_idx, class_name, id_to_path):
    path = id_to_path[img_id]
    input_tensor, img_np = load_image_for_cam(path)
    targets = [ClassifierOutputTarget(class_idx)]
    grayscale_cam = cam(input_tensor=input_tensor, targets=targets)[0, :]

    cam_cache.setdefault(img_id, {})[class_name] = grayscale_cam
    img_cache[img_id] = img_np

    bone_channel = img_np[:, :, 2]
    corr = bone_shortcut_correlation(grayscale_cam, bone_channel)
    bone_corr_records.append({"img_id": img_id, "class": class_name, "bone_corr": corr})

    visualization = show_cam_on_image(img_np, grayscale_cam, use_rgb=True)
    return img_np, visualization, grayscale_cam, corr


def show_and_save_gradcam_panel(img_id, class_idx, class_name, id_to_path, fig_axes, tag=""):
    img_np, visualization, grayscale_cam, corr = compute_gradcam(
        img_id, class_idx, class_name, id_to_path
    )
    ax_brain, ax_subdural, ax_bone, ax_composite, ax_cam = fig_axes

    ax_brain.imshow(img_np[:, :, 0], cmap="gray"); ax_brain.set_title("brain", fontsize=8); ax_brain.axis("off")
    ax_subdural.imshow(img_np[:, :, 1], cmap="gray"); ax_subdural.set_title("subdural", fontsize=8); ax_subdural.axis("off")
    ax_bone.imshow(img_np[:, :, 2], cmap="gray"); ax_bone.set_title("bone", fontsize=8); ax_bone.axis("off")
    ax_composite.imshow(img_np); ax_composite.set_title(f"{img_id}\ncomposite", fontsize=8); ax_composite.axis("off")
    ax_cam.imshow(visualization); ax_cam.set_title(f"{class_name}{tag}\nbone-corr={corr:.2f}", fontsize=8); ax_cam.axis("off")

    out_path = os.path.join(GRADCAM_OUT_DIR, f"{class_name}_{img_id}{tag.replace(' ', '_')}.png")
    Image.fromarray(visualization).save(out_path)


# ------------------------------------------------------------------
# Per-class predictions using val-derived thresholds
# ------------------------------------------------------------------
test_ids_array = test_df.index.values
class_preds = {}
for class_name in results_df.index:
    class_idx = LABEL_COLS.index(class_name)
    thresh = results_df.loc[class_name, "threshold_from_val"]
    class_preds[class_name] = (test_probs[:, class_idx] >= thresh)

# ------------------------------------------------------------------
# Compute + cache CAMs for N_CAM_SAMPLES confirmed TPs per class.
# Only the first N_VISUALIZE_PER_CLASS of each are plotted.
# ------------------------------------------------------------------
classes_to_show = ["any", "epidural", "intraparenchymal",
                    "intraventricular", "subarachnoid", "subdural"]

n_cols = N_VISUALIZE_PER_CLASS * 5
fig, axes = plt.subplots(len(classes_to_show), n_cols,
                          figsize=(3 * n_cols / 2, 3.2 * len(classes_to_show)))

sample_summary = []

for row, class_name in enumerate(classes_to_show):
    class_idx = LABEL_COLS.index(class_name)
    y_true = test_labels[:, class_idx]
    y_pred = class_preds[class_name]

    true_pos_mask = (y_true == 1) & (y_pred == 1)
    true_pos_ids = test_ids_array[true_pos_mask]
    true_pos_ids = np.array([i for i in true_pos_ids if i in id_to_path])

    n_available = len(true_pos_ids)
    n_sample = min(N_CAM_SAMPLES, n_available)
    sample_summary.append({"class": class_name, "n_available": n_available, "n_sampled": n_sample})

    if n_sample == 0:
        for col in range(n_cols):
            axes[row, col].set_visible(False)
        print(f"WARNING: no confirmed true positives with resolvable paths for '{class_name}'.")
        continue

    sample_ids = np.random.choice(true_pos_ids, size=n_sample, replace=False)

    print(f"[{class_name}] computing CAMs for {n_sample} images "
          f"({n_available} available)...")
    for i, img_id in enumerate(tqdm(sample_ids, desc=class_name, leave=False)):
        if i < N_VISUALIZE_PER_CLASS:
            panel_axes = axes[row, i * 5:(i + 1) * 5]
            show_and_save_gradcam_panel(img_id, class_idx, class_name, id_to_path,
                                         panel_axes, tag=" (confirmed TP)")
        else:
            # compute + cache only, no plotting
            compute_gradcam(img_id, class_idx, class_name, id_to_path)

    # hide unused visualization columns if fewer than N_VISUALIZE_PER_CLASS were available
    for col in range(min(N_VISUALIZE_PER_CLASS, n_sample) * 5, n_cols):
        axes[row, col].set_visible(False)

plt.suptitle(f"Grad-CAM — {MODEL_NAME}, sample visualization\n"
             f"(full cache: up to {N_CAM_SAMPLES}/class, cam_cache size = {len(cam_cache)})",
             y=1.001)
plt.tight_layout()
plt.savefig(f"/kaggle/working/{MODEL_NAME}_gradcam_sample_visualization.png",
            dpi=200, bbox_inches="tight")
plt.show()

print(f"\nSample sizes used per class:")
print(pd.DataFrame(sample_summary).to_string(index=False))

# ------------------------------------------------------------------
# Persist cam_cache + img_cache to disk — AOPC/Max-Sensitivity cell
# can load these directly instead of recomputing.
# ------------------------------------------------------------------
with open(CAM_CACHE_PATH, "wb") as f:
    pickle.dump({"cam_cache": cam_cache, "img_cache": img_cache}, f)
print(f"\nSaved cam_cache + img_cache ({len(cam_cache)} images) to {CAM_CACHE_PATH}")

# ------------------------------------------------------------------
# Bone-channel shortcut summary (now over the full sampled set)
# ------------------------------------------------------------------
bone_corr_df = pd.DataFrame(bone_corr_records)
print("\n" + "=" * 70)
print("BONE-CHANNEL SHORTCUT CHECK (full sampled set)")
print("=" * 70)
print(bone_corr_df.groupby("class")["bone_corr"].agg(["mean", "std", "count"]).round(3))

bone_corr_out = f"/kaggle/working/{MODEL_NAME}_bone_shortcut_correlations.csv"
bone_corr_df.to_csv(bone_corr_out, index=False)
print(f"\nSaved per-example bone-correlation records to {bone_corr_out}")

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torchvision import models, transforms

# ------------------------------------------------------------------
# CONFIG — run this cell first, every session, right after switching
# accelerators / restarting the kernel. Defines everything lightweight
# that every later cell (rebuild, eval, Grad-CAM, AOPC) depends on.
# ------------------------------------------------------------------
MODEL_NAME = "resnet50"   # "resnet50" or "vit_b16"

INPUT_DIRS = [
    "/kaggle/input/datasets/anushakirand/rsna-ich-preprocessed-negative-part1",
    "/kaggle/input/datasets/anushakirand/rsna-ich-preprocessed-negative-part2",
    "/kaggle/input/datasets/anushakirand/rsna-ich-preprocessed-negative-part3",
    "/kaggle/input/datasets/anushakirand/rsna-ich-preprocessed-pngs-positive-part-1",
    "/kaggle/input/datasets/anushakirand/rsna-ich-preprocessed-pngs-part3",
    "/kaggle/input/datasets/anushakirand/rsna-ich-preprocessed-pngs",
]

TRAIN_CSV = ("/kaggle/input/competitions/rsna-intracranial-hemorrhage-detection/"
             "rsna-intracranial-hemorrhage-detection/stage_2_train.csv")

MIN_FILE_SIZE_KB = 40
MIN_FILE_SIZE_BYTES = MIN_FILE_SIZE_KB * 1024

LABEL_COLS = ["any", "epidural", "intraparenchymal",
              "intraventricular", "subarachnoid", "subdural"]

BATCH_SIZE = 32
NUM_EPOCHS = 15
LR = 1e-4

VAL_SPLIT = 0.15
TEST_SPLIT = 0.15

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"

EARLY_STOPPING_PATIENCE = 5
CHECKPOINT_DIR = "/kaggle/working/checkpoints"
CHECKPOINT_PATH = f"{CHECKPOINT_DIR}/{MODEL_NAME}_best.pt"
TEST_IDS_PATH = "/kaggle/working/test_ids.csv"

NUM_WORKERS = 4
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


def build_resnet50(num_classes=len(LABEL_COLS)):
    model = models.resnet50(weights=None)
    model.fc = nn.Sequential(nn.Dropout(0.3), nn.Linear(model.fc.in_features, num_classes))
    return model.to(DEVICE)


def build_vit_b16(num_classes=len(LABEL_COLS)):
    model = models.vit_b_16(weights=None)
    model.heads = nn.Sequential(nn.Dropout(0.3), nn.Linear(model.hidden_dim, num_classes))
    return model.to(DEVICE)


MODEL_BUILDERS = {"resnet50": build_resnet50, "vit_b16": build_vit_b16}

print(f"Config loaded. Device: {DEVICE} | AMP: {USE_AMP} | Model: {MODEL_NAME}")
print("Ready for: loader cell -> eval cell -> Grad-CAM cell -> AOPC cell, in that order.")

In [ ]:
import numpy as np
import pandas as pd
import torch
from PIL import Image
from tqdm.auto import tqdm

# ------------------------------------------------------------------
# AOPC + MAX-SENSITIVITY — ResNet50, Grad-CAM explainability validation
# ------------------------------------------------------------------
# AOPC (Samek et al., 2016): measures whether the CAM correctly
# identifies pixels the model actually relies on. Progressively
# perturbs the highest-ranked CAM regions and measures how fast the
# predicted probability drops. A steep drop = a faithful explanation.
#
# Max-Sensitivity (Yeh et al., 2019): measures explanation robustness.
# Adds small random noise to the input several times and recomputes
# the CAM each time; reports the MAX heatmap change observed. Low
# sensitivity = a robust/trustworthy explanation.
#
# Perturbation baseline: PER-CHANNEL MEAN of the image itself, not
# zero-fill. Justified by the 3-window preprocessing — 0 in a given
# window channel means "at/below that window's HU lower bound," a
# specific clinical value, not "no information." Mean-fill avoids
# injecting that bias. Perturbation operates on a 14x14 patch grid
# (16px patches) rather than individual pixels, matching the native
# resolution of layer4's feature map after upsampling — perturbing
# single pixels at CAM's true resolution is not meaningful.

N_SAMPLES_PER_CLASS = 50
N_AOPC_STEPS = 10            # cumulative 10%, 20%, ..., 100% of patches perturbed
GRID_SIZE = 16               # 16x16 grid over 224x224 -> 14px patches
N_SENSITIVITY_REPEATS = 10
NOISE_STD_FRACTION = 0.05    # noise magnitude as a fraction of per-channel image std

np.random.seed(RANDOM_SEED)
patch_size = 224 // GRID_SIZE


def normalize_np_to_tensor(img_np):
    """img_np: [H,W,3] float in [0,1] -> normalized [1,3,H,W] tensor, matching eval_transform."""
    mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
    std = np.array([0.229, 0.224, 0.225], dtype=np.float32)
    normed = (img_np - mean) / std
    tensor = torch.from_numpy(normed.transpose(2, 0, 1)).unsqueeze(0).float().to(DEVICE)
    return tensor


@torch.no_grad()
def get_class_prob(img_np, class_idx):
    input_tensor = normalize_np_to_tensor(img_np)
    with torch.autocast(device_type=DEVICE.type, enabled=USE_AMP):
        outputs = model(input_tensor)
    return torch.sigmoid(outputs.float())[0, class_idx].item()


def cam_to_patch_ranking(grayscale_cam):
    """Average CAM value per grid patch, return patch indices ranked descending by importance."""
    patch_scores = np.zeros((GRID_SIZE, GRID_SIZE))
    for i in range(GRID_SIZE):
        for j in range(GRID_SIZE):
            patch = grayscale_cam[i * patch_size:(i + 1) * patch_size,
                                   j * patch_size:(j + 1) * patch_size]
            patch_scores[i, j] = patch.mean()
    flat_order = np.argsort(-patch_scores.flatten())  # descending importance
    return flat_order


def perturb_patches(img_np, patch_indices_to_remove):
    """Replace given patches (flat indices into GRID_SIZE x GRID_SIZE) with per-channel mean-fill."""
    perturbed = img_np.copy()
    channel_means = img_np.reshape(-1, 3).mean(axis=0)  # per-channel mean over whole image
    for flat_idx in patch_indices_to_remove:
        i, j = divmod(flat_idx, GRID_SIZE)
        perturbed[i * patch_size:(i + 1) * patch_size,
                  j * patch_size:(j + 1) * patch_size, :] = channel_means
    return perturbed


def compute_aopc(img_np, grayscale_cam, class_idx):
    ranking = cam_to_patch_ranking(grayscale_cam)
    total_patches = GRID_SIZE * GRID_SIZE
    original_prob = get_class_prob(img_np, class_idx)

    drops = []
    for step in range(1, N_AOPC_STEPS + 1):
        n_remove = int(total_patches * step / N_AOPC_STEPS)
        perturbed_img = perturb_patches(img_np, ranking[:n_remove])
        perturbed_prob = get_class_prob(perturbed_img, class_idx)
        drops.append(original_prob - perturbed_prob)

    return float(np.mean(drops))


def compute_max_sensitivity(img_id, class_idx, class_name, id_to_path):
    """Recomputes the CAM under small random noise perturbations; returns max heatmap change."""
    path = id_to_path[img_id]
    orig_tensor, orig_img_np = load_image_for_cam(path)
    from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
    targets = [ClassifierOutputTarget(class_idx)]

    orig_cam = cam(input_tensor=orig_tensor, targets=targets)[0, :]
    noise_std = NOISE_STD_FRACTION * orig_img_np.std()

    max_diff = 0.0
    for _ in range(N_SENSITIVITY_REPEATS):
        noise = np.random.normal(0, noise_std, orig_img_np.shape).astype(np.float32)
        noisy_img = np.clip(orig_img_np + noise, 0, 1)
        noisy_tensor = normalize_np_to_tensor(noisy_img)
        noisy_cam = cam(input_tensor=noisy_tensor, targets=targets)[0, :]
        diff = np.linalg.norm(orig_cam - noisy_cam)
        max_diff = max(max_diff, diff)

    return float(max_diff)


# ------------------------------------------------------------------
# Run over confirmed true positives per class
# ------------------------------------------------------------------
test_ids_array = test_df.index.values
records = []

for class_name in results_df.index:
    class_idx = LABEL_COLS.index(class_name)
    y_true = test_labels[:, class_idx]
    y_pred = class_preds[class_name]

    true_pos_mask = (y_true == 1) & (y_pred == 1)
    true_pos_ids = test_ids_array[true_pos_mask]
    true_pos_ids = np.array([i for i in true_pos_ids if i in id_to_path])

    n_sample = min(N_SAMPLES_PER_CLASS, len(true_pos_ids))
    if n_sample == 0:
        print(f"Skipping '{class_name}' — no confirmed true positives with resolvable paths.")
        continue

    sample_ids = np.random.choice(true_pos_ids, size=n_sample, replace=False)
    print(f"\n{class_name}: running AOPC + Max-Sensitivity on {n_sample} confirmed TPs "
          f"(available: {len(true_pos_ids)})")

    for img_id in tqdm(sample_ids, desc=class_name):
        path = id_to_path[img_id]
        _, img_np = load_image_for_cam(path)

        # reuse cached CAM if available, else compute fresh
        if img_id in cam_cache and class_name in cam_cache[img_id]:
            grayscale_cam = cam_cache[img_id][class_name]
        else:
            input_tensor, _ = load_image_for_cam(path)
            from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
            grayscale_cam = cam(input_tensor=input_tensor,
                                 targets=[ClassifierOutputTarget(class_idx)])[0, :]
            cam_cache.setdefault(img_id, {})[class_name] = grayscale_cam

        aopc = compute_aopc(img_np, grayscale_cam, class_idx)
        max_sens = compute_max_sensitivity(img_id, class_idx, class_name, id_to_path)

        records.append({
            "img_id": img_id, "class": class_name,
            "aopc": aopc, "max_sensitivity": max_sens,
        })

aopc_sens_df = pd.DataFrame(records)

print("\n" + "=" * 80)
print("AOPC + MAX-SENSITIVITY SUMMARY (mean ± std, per class)")
print("=" * 80)
summary = aopc_sens_df.groupby("class").agg(
    aopc_mean=("aopc", "mean"), aopc_std=("aopc", "std"),
    sens_mean=("max_sensitivity", "mean"), sens_std=("max_sensitivity", "std"),
    n=("aopc", "count"),
).round(4)
print(summary.to_string())

out_path = f"/kaggle/working/{MODEL_NAME}_aopc_max_sensitivity.csv"
aopc_sens_df.to_csv(out_path, index=False)
summary_out_path = f"/kaggle/working/{MODEL_NAME}_aopc_max_sensitivity_summary.csv"
summary.to_csv(summary_out_path)
print(f"\nSaved per-image results to {out_path}")
print(f"Saved per-class summary to {summary_out_path}")

AOPC and Max-Sensitivity were computed for 50 confirmed true positives per class (capped at 127 for epidural, its full confirmed-TP pool). Epidural showed both the lowest mean AOPC (0.114 ± 0.059) and among the highest Max-Sensitivity (32.6 ± 9.4) of all subtypes, indicating that even on correctly classified cases, the model's explanations for epidural are less faithful to its decision process and less robust to input perturbation than for better-performing subtypes. This corroborates the classification-level finding (Section X) and the qualitative observation of a flat, spatially undifferentiated CAM on at least one epidural false negative (Figure X), suggesting the model's difficulty with epidural reflects a genuine representational limitation rather than a thresholding artifact alone.

In [ ]:
import os
import pickle
import numpy as np
import torch
import torch.nn as nn
from torchvision import models

# ------------------------------------------------------------------
# MINIMAL RELOAD — no eval rerun, no rescan, no training. Just enough
# state for HiResCAM: model weights + the cached CAM images.
# ------------------------------------------------------------------
MODEL_NAME = "resnet50"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
LABEL_COLS = ["any", "epidural", "intraparenchymal",
              "intraventricular", "subarachnoid", "subdural"]

def build_resnet50(num_classes=len(LABEL_COLS)):
    model = models.resnet50(weights=None)
    model.fc = nn.Sequential(nn.Dropout(0.3), nn.Linear(model.fc.in_features, num_classes))
    return model.to(DEVICE)

model = build_resnet50()
checkpoint = torch.load(f"/kaggle/working/checkpoints/{MODEL_NAME}_best.pt",
                         map_location=DEVICE, weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()
print(f"Loaded checkpoint: epoch {checkpoint['epoch']}, val_loss={checkpoint['val_loss']:.4f}")

with open("/kaggle/working/cam_cache.pkl", "rb") as f:
    _cached = pickle.load(f)
cam_cache, img_cache = _cached["cam_cache"], _cached["img_cache"]
print(f"Loaded cam_cache + img_cache ({len(cam_cache)} images) from disk.")

In [ ]:
import os
import json
import pickle
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image

# ------------------------------------------------------------------
# ONE-CELL RELOAD — everything the HiResCAM + AOPC cells need,
# reconstructed from disk. No eval rerun, no rescan, no training.
# ------------------------------------------------------------------
MODEL_NAME = "resnet50"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
LABEL_COLS = ["any", "epidural", "intraparenchymal",
              "intraventricular", "subarachnoid", "subdural"]

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

def load_image_for_cam(path):
    img = Image.open(path).convert("RGB").resize((224, 224))
    img_np = np.array(img).astype(np.float32) / 255.0
    input_tensor = eval_transform(img).unsqueeze(0).to(DEVICE)
    return input_tensor, img_np

# --- model ---
def build_resnet50(num_classes=len(LABEL_COLS)):
    model = models.resnet50(weights=None)
    model.fc = nn.Sequential(nn.Dropout(0.3), nn.Linear(model.fc.in_features, num_classes))
    return model.to(DEVICE)

model = build_resnet50()
checkpoint = torch.load(f"/kaggle/working/checkpoints/{MODEL_NAME}_best.pt",
                         map_location=DEVICE, weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()
print(f"Loaded checkpoint: epoch {checkpoint['epoch']}, val_loss={checkpoint['val_loss']:.4f}")

# --- metrics + cached probs/labels ---
results_df = pd.read_csv(f"/kaggle/working/{MODEL_NAME}_eval_metrics.csv", index_col="class")
cache = np.load("/kaggle/working/test_inference_cache.npz")
val_probs, val_labels = cache["val_probs"], cache["val_labels"]
test_probs, test_labels = cache["test_probs"], cache["test_labels"]
print(f"Loaded results_df + inference arrays (test: {len(test_probs)} images)")

# --- test_df ---
TRAIN_CSV = ("/kaggle/input/competitions/rsna-intracranial-hemorrhage-detection/"
             "rsna-intracranial-hemorrhage-detection/stage_2_train.csv")

def load_labels(csv_path):
    y = pd.read_csv(csv_path)
    id_split = y.ID.str.rsplit("_", n=1, expand=True)
    y = pd.concat([id_split, y.Label], axis=1)
    y.columns = ["id", "sub_type", "label"]
    y = y.drop_duplicates(subset=["id", "sub_type"])
    return y.pivot(index="id", columns="sub_type", values="label")

test_ids = pd.read_csv("/kaggle/working/test_ids.csv")["id"].values
full_labels = load_labels(TRAIN_CSV)
test_df = full_labels.loc[test_ids]
test_ids_array = test_df.index.values
print(f"Rebuilt test_df: {len(test_df)} rows")

# --- id_to_path ---
with open("/kaggle/working/test_id_to_path.json") as f:
    id_to_path = json.load(f)
print(f"Loaded id_to_path: {len(id_to_path)} images")

# --- class_preds ---
class_preds = {}
for class_name in results_df.index:
    class_idx = LABEL_COLS.index(class_name)
    thresh = results_df.loc[class_name, "threshold_from_val"]
    class_preds[class_name] = (test_probs[:, class_idx] >= thresh)

# --- cam_cache / img_cache from Grad-CAM's pickle ---
with open("/kaggle/working/cam_cache.pkl", "rb") as f:
    _cached = pickle.load(f)
cam_cache, img_cache = _cached["cam_cache"], _cached["img_cache"]
print(f"Loaded cam_cache + img_cache ({len(cam_cache)} images)")

print("\nReady. Now re-run the HiResCAM cell in full, then the AOPC-for-HiResCAM cell.")

In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm
from pytorch_grad_cam import HiResCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image

# ------------------------------------------------------------------
# HiResCAM — reuses the exact images already sampled + decoded by the
# Grad-CAM cell (img_cache), so this is a like-for-like comparison,
# not a fresh random draw. Runs BATCHED: one forward+backward pass per
# batch of images instead of one pass per image, since target_layers
# is fixed and ClassifierOutputTarget can be supplied per-example
# across a stacked batch tensor.
# ------------------------------------------------------------------
assert "cam_cache" in globals() and "img_cache" in globals(), \
    "Run the Grad-CAM cell first — this reuses its cached images."

HIRESCAM_OUT_DIR = "/kaggle/working/hirescam_examples"
HIRESCAM_CACHE_PATH = "/kaggle/working/hirescam_cache.pkl"
os.makedirs(HIRESCAM_OUT_DIR, exist_ok=True)

BATCH_SIZE_CAM = 16     # tune to GPU memory — this is what makes it non-naive
N_VISUALIZE_PER_CLASS = 2

mean_arr = np.array([0.485, 0.456, 0.406], dtype=np.float32)
std_arr = np.array([0.229, 0.224, 0.225], dtype=np.float32)

hirescam_target_layers = [model.layer4[-1]]
hirescam = HiResCAM(model=model, target_layers=hirescam_target_layers)

hirescam_cache = {}              # img_id -> {class_name: grayscale_cam}
hirescam_bone_corr_records = []


def img_np_to_tensor(img_np):
    normed = (img_np - mean_arr) / std_arr
    return torch.from_numpy(normed.transpose(2, 0, 1)).float()


def batched_hirescam(img_ids, class_idx, class_name):
    """One forward+backward pass for the whole batch, not one per image."""
    tensors = torch.stack([img_np_to_tensor(img_cache[i]) for i in img_ids]).to(DEVICE)
    targets = [ClassifierOutputTarget(class_idx) for _ in img_ids]
    grayscale_cams = hirescam(input_tensor=tensors, targets=targets)  # [B, H, W]

    for i, img_id in enumerate(img_ids):
        gc = grayscale_cams[i]
        hirescam_cache.setdefault(img_id, {})[class_name] = gc

        bone_channel = img_cache[img_id][:, :, 2]
        cam_flat, bone_flat = gc.flatten(), bone_channel.flatten()
        corr = (float(np.corrcoef(cam_flat, bone_flat)[0, 1])
                if cam_flat.std() > 1e-8 and bone_flat.std() > 1e-8 else np.nan)
        hirescam_bone_corr_records.append(
            {"img_id": img_id, "class": class_name, "bone_corr": corr}
        )
    return grayscale_cams


# ------------------------------------------------------------------
# Walk the same img_id -> class_name pairs already present in cam_cache
# (from the Grad-CAM cell), batched per class instead of one-by-one.
# ------------------------------------------------------------------
classes_to_show = ["any", "epidural", "intraparenchymal",
                    "intraventricular", "subarachnoid", "subdural"]

viz_examples = {}  # class_name -> list of (img_id, visualization) for the sanity panel

for class_name in classes_to_show:
    img_ids_for_class = [img_id for img_id, per_class in cam_cache.items()
                          if class_name in per_class]
    if not img_ids_for_class:
        print(f"Skipping '{class_name}' — no cached images for this class.")
        continue

    class_idx = LABEL_COLS.index(class_name)
    print(f"[{class_name}] running HiResCAM on {len(img_ids_for_class)} cached images "
          f"in batches of {BATCH_SIZE_CAM}...")

    all_gcs = []
    for start in tqdm(range(0, len(img_ids_for_class), BATCH_SIZE_CAM),
                       desc=class_name, leave=False):
        batch_ids = img_ids_for_class[start:start + BATCH_SIZE_CAM]
        gcs = batched_hirescam(batch_ids, class_idx, class_name)
        all_gcs.extend(zip(batch_ids, gcs))

    viz_examples[class_name] = all_gcs[:N_VISUALIZE_PER_CLASS]

# ------------------------------------------------------------------
# Small sanity-check visualization panel (same layout style as Grad-CAM cell)
# ------------------------------------------------------------------
n_cols = N_VISUALIZE_PER_CLASS * 5
rows_with_examples = [c for c in classes_to_show if c in viz_examples]
fig, axes = plt.subplots(len(rows_with_examples), n_cols,
                          figsize=(3 * n_cols / 2, 3.2 * len(rows_with_examples)))
if len(rows_with_examples) == 1:
    axes = axes[np.newaxis, :]

for row, class_name in enumerate(rows_with_examples):
    for i, (img_id, gc) in enumerate(viz_examples[class_name]):
        img_np = img_cache[img_id]
        visualization = show_cam_on_image(img_np, gc, use_rgb=True)
        Image.fromarray(visualization).save(
            os.path.join(HIRESCAM_OUT_DIR, f"{class_name}_{img_id}_hirescam.png")
        )

        ax_brain, ax_subdural, ax_bone, ax_composite, ax_cam = axes[row, i * 5:(i + 1) * 5]
        ax_brain.imshow(img_np[:, :, 0], cmap="gray"); ax_brain.set_title("brain", fontsize=8); ax_brain.axis("off")
        ax_subdural.imshow(img_np[:, :, 1], cmap="gray"); ax_subdural.set_title("subdural", fontsize=8); ax_subdural.axis("off")
        ax_bone.imshow(img_np[:, :, 2], cmap="gray"); ax_bone.set_title("bone", fontsize=8); ax_bone.axis("off")
        ax_composite.imshow(img_np); ax_composite.set_title(f"{img_id}\ncomposite", fontsize=8); ax_composite.axis("off")
        ax_cam.imshow(visualization); ax_cam.set_title(f"{class_name} (HiResCAM)", fontsize=8); ax_cam.axis("off")

    for col in range(len(viz_examples[class_name]) * 5, n_cols):
        axes[row, col].set_visible(False)

plt.suptitle(f"HiResCAM — {MODEL_NAME}, sample visualization (batched, reused cache)", y=1.001)
plt.tight_layout()
plt.savefig(f"/kaggle/working/{MODEL_NAME}_hirescam_sample_visualization.png",
            dpi=200, bbox_inches="tight")
plt.show()

# ------------------------------------------------------------------
# Persist cache + bone-shortcut correlations
# ------------------------------------------------------------------
with open(HIRESCAM_CACHE_PATH, "wb") as f:
    pickle.dump({"hirescam_cache": hirescam_cache}, f)
print(f"\nSaved hirescam_cache ({len(hirescam_cache)} images) to {HIRESCAM_CACHE_PATH}")

hirescam_bone_corr_df = pd.DataFrame(hirescam_bone_corr_records)
print("\n" + "=" * 70)
print("BONE-CHANNEL SHORTCUT CHECK — HiResCAM")
print("=" * 70)
print(hirescam_bone_corr_df.groupby("class")["bone_corr"].agg(["mean", "std", "count"]).round(3))

hirescam_bone_corr_out = f"/kaggle/working/{MODEL_NAME}_hirescam_bone_shortcut_correlations.csv"
hirescam_bone_corr_df.to_csv(hirescam_bone_corr_out, index=False)
print(f"\nSaved per-example bone-correlation records to {hirescam_bone_corr_out}")

In [ ]:
import os
import pickle
import pandas as pd

HIRESCAM_CACHE_PATH = "/kaggle/working/hirescam_cache.pkl"
HIRESCAM_BONE_CORR_PATH = "/kaggle/working/resnet50_hirescam_bone_shortcut_correlations.csv"

# --- Check 1: hirescam_cache.pkl ---
if os.path.exists(HIRESCAM_CACHE_PATH):
    with open(HIRESCAM_CACHE_PATH, "rb") as f:
        _cached = pickle.load(f)
    n_images = len(_cached.get("hirescam_cache", {}))
    print(f"FOUND: {HIRESCAM_CACHE_PATH}")
    print(f"  -> hirescam_cache contains {n_images} images")
else:
    print(f"MISSING: {HIRESCAM_CACHE_PATH}")

print()

# --- Check 2: bone-correlation CSV ---
if os.path.exists(HIRESCAM_BONE_CORR_PATH):
    df = pd.read_csv(HIRESCAM_BONE_CORR_PATH)
    print(f"FOUND: {HIRESCAM_BONE_CORR_PATH}")
    print(f"  -> {len(df)} rows")
    print(df.groupby("class")["bone_corr"].agg(["mean", "std", "count"]).round(3))
else:
    print(f"MISSING: {HIRESCAM_BONE_CORR_PATH}")

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import torch
from torchvision import transforms

MODEL_NAME = "resnet50"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
RANDOM_SEED = 42
LABEL_COLS = ["any", "epidural", "intraparenchymal",
              "intraventricular", "subarachnoid", "subdural"]

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# --- metrics + cached probs/labels ---
results_df = pd.read_csv(f"/kaggle/working/{MODEL_NAME}_eval_metrics.csv", index_col="class")
cache = np.load("/kaggle/working/test_inference_cache.npz")
val_probs, val_labels = cache["val_probs"], cache["val_labels"]
test_probs, test_labels = cache["test_probs"], cache["test_labels"]
print(f"Loaded results_df + inference arrays (test: {len(test_probs)} images)")

# --- test_df (labels only, for indexing) ---
TRAIN_CSV = ("/kaggle/input/competitions/rsna-intracranial-hemorrhage-detection/"
             "rsna-intracranial-hemorrhage-detection/stage_2_train.csv")

def load_labels(csv_path):
    y = pd.read_csv(csv_path)
    id_split = y.ID.str.rsplit("_", n=1, expand=True)
    y = pd.concat([id_split, y.Label], axis=1)
    y.columns = ["id", "sub_type", "label"]
    y = y.drop_duplicates(subset=["id", "sub_type"])
    return y.pivot(index="id", columns="sub_type", values="label")

test_ids = pd.read_csv("/kaggle/working/test_ids.csv")["id"].values
full_labels = load_labels(TRAIN_CSV)
test_df = full_labels.loc[test_ids]
print(f"Rebuilt test_df: {len(test_df)} rows")

# --- id_to_path (cached, no rescan) ---
with open("/kaggle/working/test_id_to_path.json") as f:
    id_to_path = json.load(f)
print(f"Loaded id_to_path: {len(id_to_path)} images")

# --- class_preds (needed by the AOPC cell's true_pos_mask logic) ---
test_ids_array = test_df.index.values
class_preds = {}
for class_name in results_df.index:
    class_idx = LABEL_COLS.index(class_name)
    thresh = results_df.loc[class_name, "threshold_from_val"]
    class_preds[class_name] = (test_probs[:, class_idx] >= thresh)

print("Ready — test_df, id_to_path, results_df, test_probs/labels, class_preds all set.")

In [ ]:
import numpy as np
import pandas as pd
import torch
from PIL import Image
from tqdm.auto import tqdm
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

# ------------------------------------------------------------------
# AOPC + MAX-SENSITIVITY — ResNet50, HiResCAM explainability validation
# ------------------------------------------------------------------
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

N_SAMPLES_PER_CLASS = 50
N_AOPC_STEPS = 10
GRID_SIZE = 16
N_SENSITIVITY_REPEATS = 10
NOISE_STD_FRACTION = 0.05

patch_size = 224 // GRID_SIZE


def normalize_np_to_tensor(img_np):
    mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
    std = np.array([0.229, 0.224, 0.225], dtype=np.float32)
    normed = (img_np - mean) / std
    tensor = torch.from_numpy(normed.transpose(2, 0, 1)).unsqueeze(0).float().to(DEVICE)
    return tensor


@torch.no_grad()
def get_class_prob(img_np, class_idx):
    input_tensor = normalize_np_to_tensor(img_np)
    with torch.autocast(device_type=DEVICE.type, enabled=USE_AMP):
        outputs = model(input_tensor)
    return torch.sigmoid(outputs.float())[0, class_idx].item()


def cam_to_patch_ranking(grayscale_cam):
    patch_scores = np.zeros((GRID_SIZE, GRID_SIZE))
    for i in range(GRID_SIZE):
        for j in range(GRID_SIZE):
            patch = grayscale_cam[i * patch_size:(i + 1) * patch_size,
                                   j * patch_size:(j + 1) * patch_size]
            patch_scores[i, j] = patch.mean()
    return np.argsort(-patch_scores.flatten())


def perturb_patches(img_np, patch_indices_to_remove):
    perturbed = img_np.copy()
    channel_means = img_np.reshape(-1, 3).mean(axis=0)
    for flat_idx in patch_indices_to_remove:
        i, j = divmod(flat_idx, GRID_SIZE)
        perturbed[i * patch_size:(i + 1) * patch_size,
                  j * patch_size:(j + 1) * patch_size, :] = channel_means
    return perturbed


def compute_aopc(img_np, grayscale_cam, class_idx):
    ranking = cam_to_patch_ranking(grayscale_cam)
    total_patches = GRID_SIZE * GRID_SIZE
    original_prob = get_class_prob(img_np, class_idx)
    drops = []
    for step in range(1, N_AOPC_STEPS + 1):
        n_remove = int(total_patches * step / N_AOPC_STEPS)
        perturbed_img = perturb_patches(img_np, ranking[:n_remove])
        perturbed_prob = get_class_prob(perturbed_img, class_idx)
        drops.append(original_prob - perturbed_prob)
    return float(np.mean(drops))


def compute_max_sensitivity(img_id, class_idx, class_name, id_to_path):
    """Recomputes the HiResCAM under small random noise perturbations; returns max heatmap change."""
    path = id_to_path[img_id]
    orig_tensor, orig_img_np = load_image_for_cam(path)
    targets = [ClassifierOutputTarget(class_idx)]

    orig_cam = hirescam(input_tensor=orig_tensor, targets=targets)[0, :]   # hirescam, not cam
    noise_std = NOISE_STD_FRACTION * orig_img_np.std()

    max_diff = 0.0
    for _ in range(N_SENSITIVITY_REPEATS):
        noise = np.random.normal(0, noise_std, orig_img_np.shape).astype(np.float32)
        noisy_img = np.clip(orig_img_np + noise, 0, 1)
        noisy_tensor = normalize_np_to_tensor(noisy_img)
        noisy_cam = hirescam(input_tensor=noisy_tensor, targets=targets)[0, :]   # hirescam, not cam
        diff = np.linalg.norm(orig_cam - noisy_cam)
        max_diff = max(max_diff, diff)

    return float(max_diff)


# ------------------------------------------------------------------
# Run over confirmed true positives per class
# ------------------------------------------------------------------
test_ids_array = test_df.index.values
records = []

for class_name in results_df.index:
    class_idx = LABEL_COLS.index(class_name)
    y_true = test_labels[:, class_idx]
    y_pred = class_preds[class_name]

    true_pos_mask = (y_true == 1) & (y_pred == 1)
    true_pos_ids = test_ids_array[true_pos_mask]
    true_pos_ids = np.array([i for i in true_pos_ids if i in id_to_path])

    n_sample = min(N_SAMPLES_PER_CLASS, len(true_pos_ids))
    if n_sample == 0:
        print(f"Skipping '{class_name}' — no confirmed true positives with resolvable paths.")
        continue

    sample_ids = np.random.choice(true_pos_ids, size=n_sample, replace=False)
    print(f"\n{class_name}: running AOPC + Max-Sensitivity (HiResCAM) on {n_sample} confirmed TPs "
          f"(available: {len(true_pos_ids)})")

    for img_id in tqdm(sample_ids, desc=class_name):
        path = id_to_path[img_id]
        _, img_np = load_image_for_cam(path)

        # reuse cached HiResCAM if available, else compute fresh
        if img_id in hirescam_cache and class_name in hirescam_cache[img_id]:   # hirescam_cache, not cam_cache
            grayscale_cam = hirescam_cache[img_id][class_name]
        else:
            input_tensor, _ = load_image_for_cam(path)
            grayscale_cam = hirescam(input_tensor=input_tensor,                # hirescam, not cam
                                 targets=[ClassifierOutputTarget(class_idx)])[0, :]
            hirescam_cache.setdefault(img_id, {})[class_name] = grayscale_cam   # hirescam_cache, not cam_cache

        aopc = compute_aopc(img_np, grayscale_cam, class_idx)
        max_sens = compute_max_sensitivity(img_id, class_idx, class_name, id_to_path)

        records.append({
            "img_id": img_id, "class": class_name,
            "aopc": aopc, "max_sensitivity": max_sens,
        })

aopc_sens_df = pd.DataFrame(records)

print("\n" + "=" * 80)
print("AOPC + MAX-SENSITIVITY SUMMARY — HiResCAM (mean ± std, per class)")
print("=" * 80)
summary = aopc_sens_df.groupby("class").agg(
    aopc_mean=("aopc", "mean"), aopc_std=("aopc", "std"),
    sens_mean=("max_sensitivity", "mean"), sens_std=("max_sensitivity", "std"),
    n=("aopc", "count"),
).round(4)
print(summary.to_string())

out_path = f"/kaggle/working/{MODEL_NAME}_aopc_hirescam_max_sensitivity.csv"
aopc_sens_df.to_csv(out_path, index=False)
summary_out_path = f"/kaggle/working/{MODEL_NAME}_aopc_hirescam_max_sensitivity_summary.csv"
summary.to_csv(summary_out_path)
print(f"\nSaved per-image results to {out_path}")
print(f"Saved per-class summary to {summary_out_path}")

In [ ]:
import subprocess
print(subprocess.run(["du", "-sh", "/kaggle/working"], capture_output=True, text=True).stdout)

In [ ]:
import os
import zipfile

# ------------------------------------------------------------------
# ZIP EVERYTHING IN /kaggle/working — except previous zip files and
# the folders that were only ever staging copies for those old zips.
# Includes checkpoints (.pt), pickles (.pkl), csvs, pngs, json, npz,
# txt — everything that isn't explicitly excluded below.
# ------------------------------------------------------------------
WORKING_DIR = "/kaggle/working"
OUTPUT_ZIP_PATH = "/kaggle/working/all_outputs_full.zip"

# Edit this list if you spot something else that shouldn't go in.
EXCLUDE_NAMES = {
    "all_outputs.zip",
    "resnet.zip",
    "resnet_eval_backup.zip",
    "resnet_eval_backup_files",   # staging folder for resnet_eval_backup.zip
    ".virtual_documents",         # Jupyter/Kaggle internal, not real output
}

# Also skip any .zip file by extension, in case there are others not listed above.
def should_skip(rel_path):
    parts = rel_path.split(os.sep)
    if parts[0] in EXCLUDE_NAMES:
        return True
    if rel_path.endswith(".zip"):
        return True
    if rel_path == os.path.basename(OUTPUT_ZIP_PATH):
        return True
    return False

included, skipped = [], []

with zipfile.ZipFile(OUTPUT_ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files in os.walk(WORKING_DIR):
        rel_root = os.path.relpath(root, WORKING_DIR)

        # prune excluded directories in-place so os.walk doesn't descend into them
        if rel_root != ".":
            top_level = rel_root.split(os.sep)[0]
            if top_level in EXCLUDE_NAMES:
                dirs[:] = []
                continue

        for fname in files:
            full_path = os.path.join(root, fname)
            rel_path = os.path.relpath(full_path, WORKING_DIR)

            if should_skip(rel_path):
                skipped.append(rel_path)
                continue

            zf.write(full_path, arcname=rel_path)
            included.append(rel_path)

print(f"Included {len(included)} files:")
for f in sorted(included):
    print(f"  + {f}")

print(f"\nSkipped {len(skipped)} files/folders:")
for f in sorted(skipped):
    print(f"  - {f}")

print(f"\nZip saved to {OUTPUT_ZIP_PATH} ({os.path.getsize(OUTPUT_ZIP_PATH) / 1e6:.1f} MB)")

In [ ]:
import os
import zipfile

OUTPUT_ZIP = "/kaggle/working/afterhighres_output.zip"

FILES_TO_ZIP = [
    "/kaggle/working/resnet50_aopc_hirescam_max_sensitivity.csv",
    "/kaggle/working/resnet50_aopc_hirescam_max_sensitivity_summary.csv",
    "/kaggle/working/hirescam_cache.pkl",
    "/kaggle/working/resnet50_hirescam_bone_shortcut_correlations.csv",
]

with zipfile.ZipFile(OUTPUT_ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
    for file_path in FILES_TO_ZIP:
        if os.path.exists(file_path):
            zf.write(file_path, arcname=os.path.basename(file_path))
            print(f"Added: {os.path.basename(file_path)}")
        else:
            print(f"Missing: {file_path}")

print(f"\nZIP created successfully!")
print(f"Location: {OUTPUT_ZIP}")
print(f"Size: {os.path.getsize(OUTPUT_ZIP) / 1e6:.2f} MB")

In [1]:
# ================================================================
# RESNET-50 — BONE-CHANNEL SHORTCUT / CORRELATION ANALYSIS
# Consistent with ViT-B/16 and ConvNeXt-Base:
#   - 30 confirmed true positives per class
#   - random_state = 42
#   - 224x224 input
#   - same ImageNet normalization
#   - Grad-CAM target = layer4[-1]
#   - Pearson correlation between CAM and bone channel
#   - NaN retained when CAM/bone channel has negligible variance
#
# IMPORTANT:
#   This does NOT retrain ResNet-50.
#   It loads the existing best checkpoint (epoch 14).
# ================================================================

# If pytorch-grad-cam is already installed, this does nothing.
!pip install grad-cam -q

import os
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from PIL import Image
from torchvision import models, transforms
from tqdm.auto import tqdm

from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget


# ================================================================
# 1. CONFIGURATION
# ================================================================

MODEL_NAME = "resnet50"

LABEL_COLS = [
    "any",
    "epidural",
    "intraparenchymal",
    "intraventricular",
    "subarachnoid",
    "subdural",
]

RANDOM_SEED = 42
N_CAM_SAMPLES = 30

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CHECKPOINT_PATH = (
    f"/kaggle/working/checkpoints/{MODEL_NAME}_best.pt"
)

EVAL_METRICS_PATH = (
    f"/kaggle/working/{MODEL_NAME}_eval_metrics.csv"
)

INFERENCE_CACHE_PATH = (
    "/kaggle/working/test_inference_cache.npz"
)

TEST_IDS_PATH = (
    "/kaggle/working/test_ids.csv"
)

ID_TO_PATH_CACHE = (
    "/kaggle/working/test_id_to_path.json"
)

OUTPUT_PATH = (
    f"/kaggle/working/{MODEL_NAME}_bone_shortcut_correlations.csv"
)

np.random.seed(RANDOM_SEED)

print(f"Device: {DEVICE}")
print(f"Sampling: up to {N_CAM_SAMPLES} confirmed TPs per class")
print(f"Random seed: {RANDOM_SEED}")


# ================================================================
# 2. CHECK REQUIRED FILES
# ================================================================

required_files = [
    CHECKPOINT_PATH,
    EVAL_METRICS_PATH,
    INFERENCE_CACHE_PATH,
    TEST_IDS_PATH,
]

missing = [p for p in required_files if not os.path.exists(p)]

if missing:
    raise FileNotFoundError(
        "The following required files are missing:\n"
        + "\n".join(f"  - {p}" for p in missing)
        + "\n\nMake sure the existing ResNet evaluation/checkpoint "
          "outputs are available in this Kaggle session."
    )

print("\nRequired files found.")


# ================================================================
# 3. LOAD EXISTING TEST PREDICTIONS / THRESHOLDS
#    No model inference is rerun here.
# ================================================================

results_df = pd.read_csv(
    EVAL_METRICS_PATH,
    index_col="class"
)

cache = np.load(INFERENCE_CACHE_PATH)

val_probs = cache["val_probs"]
val_labels = cache["val_labels"]
test_probs = cache["test_probs"]
test_labels = cache["test_labels"]

print(
    f"\nLoaded cached inference:"
    f"\n  validation: {len(val_probs)}"
    f"\n  test:       {len(test_probs)}"
)

print("\nUsing validation-derived thresholds:")
print(
    results_df[["threshold_from_val"]]
    .round(6)
    .to_string()
)


# ================================================================
# 4. REBUILD TEST ID TABLE
# ================================================================

TRAIN_CSV = (
    "/kaggle/input/competitions/"
    "rsna-intracranial-hemorrhage-detection/"
    "rsna-intracranial-hemorrhage-detection/"
    "stage_2_train.csv"
)


def load_labels(csv_path):
    y = pd.read_csv(csv_path)

    id_split = y.ID.str.rsplit("_", n=1, expand=True)

    y = pd.concat(
        [id_split, y.Label],
        axis=1
    )

    y.columns = ["id", "sub_type", "label"]

    y = y.drop_duplicates(
        subset=["id", "sub_type"]
    )

    return y.pivot(
        index="id",
        columns="sub_type",
        values="label"
    )


test_ids = pd.read_csv(TEST_IDS_PATH)["id"].values

full_labels = load_labels(TRAIN_CSV)

test_df = full_labels.loc[test_ids]

print(
    f"\nRebuilt test dataframe: {len(test_df)} images"
)


# ================================================================
# 5. RESOLVE IMAGE PATHS
#    Reuse cached mapping if available.
# ================================================================

if os.path.exists(ID_TO_PATH_CACHE):

    with open(ID_TO_PATH_CACHE, "r") as f:
        id_to_path = json.load(f)

    # Keep only IDs actually present in this test set.
    id_to_path = {
        img_id: path
        for img_id, path in id_to_path.items()
        if img_id in set(test_df.index)
        and os.path.exists(path)
    }

    print(
        f"Loaded cached image paths: "
        f"{len(id_to_path)} / {len(test_df)}"
    )

else:

    INPUT_DIRS = [
        "/kaggle/input/datasets/anushakirand/"
        "rsna-ich-preprocessed-negative-part1",

        "/kaggle/input/datasets/anushakirand/"
        "rsna-ich-preprocessed-negative-part2",

        "/kaggle/input/datasets/anushakirand/"
        "rsna-ich-preprocessed-negative-part3",

        "/kaggle/input/datasets/anushakirand/"
        "rsna-ich-preprocessed-pngs-positive-part-1",

        "/kaggle/input/datasets/anushakirand/"
        "rsna-ich-preprocessed-pngs-part3",

        "/kaggle/input/datasets/anushakirand/"
        "rsna-ich-preprocessed-pngs",
    ]

    needed_ids = set(test_df.index)

    id_to_path = {}

    for directory in INPUT_DIRS:

        if not os.path.isdir(directory):
            print(
                f"WARNING: directory not found: {directory}"
            )
            continue

        for filename in os.listdir(directory):

            if not filename.endswith(".png"):
                continue

            img_id = filename[:-4]

            if img_id in needed_ids:
                id_to_path[img_id] = os.path.join(
                    directory,
                    filename
                )

    with open(ID_TO_PATH_CACHE, "w") as f:
        json.dump(id_to_path, f)

    print(
        f"Resolved image paths: "
        f"{len(id_to_path)} / {len(test_df)}"
    )


if len(id_to_path) == 0:
    raise RuntimeError(
        "No test image paths could be resolved."
    )


# ================================================================
# 6. BUILD + LOAD THE EXISTING RESNET-50 CHECKPOINT
# ================================================================

def build_resnet50(
    num_classes=len(LABEL_COLS)
):
    model = models.resnet50(
        weights=None
    )

    model.fc = nn.Sequential(
        nn.Dropout(0.3),
        nn.Linear(
            model.fc.in_features,
            num_classes
        )
    )

    return model.to(DEVICE)


model = build_resnet50()

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=DEVICE,
    weights_only=False
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model.eval()

print(
    "\nLoaded ResNet-50 checkpoint:"
    f"\n  epoch:    {checkpoint['epoch']}"
    f"\n  val loss: {checkpoint['val_loss']:.4f}"
)

assert checkpoint["epoch"] == 14, (
    f"Expected ResNet best checkpoint at epoch 14, "
    f"but loaded epoch {checkpoint['epoch']}."
)


# ================================================================
# 7. SAME EVALUATION TRANSFORM
# ================================================================

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])


# ================================================================
# 8. GRAD-CAM SETUP
#    Same target layer used in the existing ResNet notebook.
# ================================================================

target_layers = [
    model.layer4[-1]
]

cam = GradCAM(
    model=model,
    target_layers=target_layers
)

print(
    "\nGrad-CAM target layer:"
    " model.layer4[-1]"
)


# ================================================================
# 9. IMAGE + BONE-CORRELATION FUNCTIONS
# ================================================================

def load_image_for_cam(path):

    img = (
        Image.open(path)
        .convert("RGB")
        .resize((224, 224))
    )

    img_np = (
        np.array(img)
        .astype(np.float32)
        / 255.0
    )

    input_tensor = (
        eval_transform(img)
        .unsqueeze(0)
        .to(DEVICE)
    )

    return input_tensor, img_np


def bone_shortcut_correlation(
    grayscale_cam,
    bone_channel
):

    cam_flat = grayscale_cam.flatten()
    bone_flat = bone_channel.flatten()

    # Same rule as the original notebook:
    # retain NaN when either vector has negligible variance.
    if (
        cam_flat.std() < 1e-8
        or bone_flat.std() < 1e-8
    ):
        return np.nan

    return float(
        np.corrcoef(
            cam_flat,
            bone_flat
        )[0, 1]
    )


# ================================================================
# 10. DETERMINE CONFIRMED TRUE POSITIVES
#     Thresholds come from validation, exactly as in the report.
# ================================================================

test_ids_array = test_df.index.values

class_preds = {}

for class_name in LABEL_COLS:

    threshold = results_df.loc[
        class_name,
        "threshold_from_val"
    ]

    class_idx = LABEL_COLS.index(
        class_name
    )

    class_preds[class_name] = (
        test_probs[:, class_idx]
        >= threshold
    )


# ================================================================
# 11. RUN BONE-CORRELATION ANALYSIS
# ================================================================

bone_corr_records = []

sample_summary = []

for class_name in LABEL_COLS:

    class_idx = LABEL_COLS.index(
        class_name
    )

    y_true = test_labels[:, class_idx]
    y_pred = class_preds[class_name]

    # Confirmed true positive:
    # ground truth positive AND model positive
    true_pos_mask = (
        (y_true == 1)
        & (y_pred == 1)
    )

    true_pos_ids = (
        test_ids_array[true_pos_mask]
    )

    # Keep only images whose files can actually be resolved.
    true_pos_ids = np.array([
        img_id
        for img_id in true_pos_ids
        if img_id in id_to_path
    ])

    n_available = len(true_pos_ids)

    n_sample = min(
        N_CAM_SAMPLES,
        n_available
    )

    sample_summary.append({
        "class": class_name,
        "confirmed_TPs_available": n_available,
        "n_sampled": n_sample,
    })

    if n_sample == 0:

        print(
            f"\nWARNING: {class_name} — "
            "no resolvable confirmed TPs."
        )

        continue

    # Same sampling protocol:
    # without replacement, random seed = 42.
    sample_ids = np.random.choice(
        true_pos_ids,
        size=n_sample,
        replace=False
    )

    print(
        f"\n{class_name}: "
        f"{n_sample} sampled / "
        f"{n_available} confirmed TPs available"
    )

    for img_id in tqdm(
        sample_ids,
        desc=class_name
    ):

        path = id_to_path[img_id]

        input_tensor, img_np = (
            load_image_for_cam(path)
        )

        targets = [
            ClassifierOutputTarget(
                class_idx
            )
        ]

        grayscale_cam = cam(
            input_tensor=input_tensor,
            targets=targets
        )[0, :]

        # Channel 2 = B = bone window
        bone_channel = img_np[:, :, 2]

        corr = bone_shortcut_correlation(
            grayscale_cam,
            bone_channel
        )

        bone_corr_records.append({
            "img_id": img_id,
            "class": class_name,
            "bone_corr": corr,
        })


# ================================================================
# 12. SAVE RAW RESULTS
# ================================================================

bone_corr_df = pd.DataFrame(
    bone_corr_records
)

bone_corr_df.to_csv(
    OUTPUT_PATH,
    index=False
)

print(
    "\nSaved raw per-example results to:"
    f"\n{OUTPUT_PATH}"
)


# ================================================================
# 13. PRINT SAMPLE COUNTS
# ================================================================

sample_summary_df = pd.DataFrame(
    sample_summary
)

print("\n" + "=" * 80)
print("SAMPLE COUNTS")
print("=" * 80)

print(
    sample_summary_df.to_string(
        index=False
    )
)


# ================================================================
# 14. SUMMARY STATISTICS FOR SECTION 6.4
# ================================================================

summary = (
    bone_corr_df
    .groupby("class")["bone_corr"]
    .agg(
        mean="mean",
        std="std",
        n_finite="count",
        n_total="size",
        min="min",
        max="max",
    )
)

summary["abs_r_gt_0.2"] = (
    bone_corr_df
    .groupby("class")["bone_corr"]
    .apply(
        lambda x: (x.abs() > 0.2).sum()
    )
)

summary["abs_r_gt_0.3"] = (
    bone_corr_df
    .groupby("class")["bone_corr"]
    .apply(
        lambda x: (x.abs() > 0.3).sum()
    )
)

summary = summary.round(4)

print("\n" + "=" * 80)
print("BONE-CHANNEL CORRELATION SUMMARY")
print("=" * 80)

print(
    summary.to_string()
)


# ================================================================
# 15. SANITY CHECK
# ================================================================

print("\n" + "=" * 80)
print("SANITY CHECK")
print("=" * 80)

for class_name in LABEL_COLS:

    class_rows = bone_corr_df[
        bone_corr_df["class"] == class_name
    ]

    print(
        f"{class_name:20s} "
        f"total={len(class_rows):2d}, "
        f"finite={class_rows['bone_corr'].notna().sum():2d}, "
        f"NaN={class_rows['bone_corr'].isna().sum():2d}"
    )

print("\nDone.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 66.3 MB/s eta 0:00:00:00:010:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Device: cuda
Sampling: up to 30 confirmed TPs per class
Random seed: 42

Required files found.

Loaded cached inference:
  validation: 40372
  test:       40373

Using validation-derived thresholds:
                  threshold_from_val
class                               
any                         0.547071
epidural                    0.088347
intraparenchymal            0.348978
intraventricular            0.369315
subarachnoid                0.340156
subdural                    0.379378

Rebuilt test dataframe: 40373 images
Loaded cached image paths: 40373 / 40373

Loaded ResNet-50 checkpoint:
  epoch:    14
  val loss: 0.1905

Grad-CAM target layer: model.layer4[-1]

any: 30 sampled / 13751 confirmed TPs available


any:   0%|          | 0/30 [00:00<?, ?it/s]


epidural: 30 sampled / 127 confirmed TPs available


epidural:   0%|          | 0/30 [00:00<?, ?it/s]


intraparenchymal: 30 sampled / 3906 confirmed TPs available


...


subarachnoid: 30 sampled / 3117 confirmed TPs available


subarachnoid:   0%|          | 0/30 [00:00<?, ?it/s]


subdural: 30 sampled / 4794 confirmed TPs available


subdural:   0%|          | 0/30 [00:00<?, ?it/s]


Saved raw per-example results to:
/kaggle/working/resnet50_bone_shortcut_correlations.csv

SAMPLE COUNTS
           class  confirmed_TPs_available  n_sampled
             any                    13751         30
        epidural                      127         30
intraparenchymal                     3906         30
intraventricular                     2941         30
    subarachnoid                     3117         30
        subdural                     4794         30

BONE-CHANNEL CORRELATION SUMMARY
                    mean     std  n_finite  n_total     min     max  abs_r_gt_0.2  abs_r_gt_0.3
class                                                                                          
any               0.0855  0.0748        30       30 -0.0466  0.2497             2             0
epidural          0.0365  0.0799        29       30 -0.1659  0.1753             0             0
intraparenchymal  0.0613  0.0482        30       30 -0.0127  0.1622             0             0
intravent

In [2]:
# Save ResNet-50 bone-channel correlation results as CSV

OUTPUT_PATH = "/kaggle/working/resnet50_bone_shortcut_correlations.csv"

bone_corr_df.to_csv(
    OUTPUT_PATH,
    index=False
)

print(f"Saved CSV to: {OUTPUT_PATH}")
print(f"Rows: {len(bone_corr_df)}")
print("\nPreview:")
display(bone_corr_df.head())

Saved CSV to: /kaggle/working/resnet50_bone_shortcut_correlations.csv
Rows: 180

Preview:


,img_id,class,bone_corr
0,ID_42f530a0e,any,0.016913
1,ID_22d442594,any,0.128215
2,ID_5261c0876,any,-0.046552
3,ID_131216d84,any,0.184034
4,ID_465d4154c,any,0.085843
